# <center>HermesAnalytics 第三节课：管理员受控变更流程与多智能体归因工作台</center>

&emsp;&emsp;前两节课已经沿可信 NL2SQL 主线走通“提出问题、生成分析计划、人工确认、安全执行、冻结证据与生成结论”。第三节课继续回答两个自然问题：当现有领域里没有学员想分析的新维度时，管理员怎样让它安全进入分析运行态；当一份可信分析已经完成后，分析师又怎样让多个角色围绕同一份冻结证据完成可追溯归因。

&emsp;&emsp;本课不按页面功能清单组织，而是沿一条连续状态主线学习：**分析能力不足 → 管理员提交结构化变更 → 确定性程序审校、编译、预演和执行 → 三处验真 → 分析师完成可信分析 → 冻结 Evidence（证据）→ Hermes Agent 承担三个受限角色 → 应用层固定工作流 → 归因报告交给人类判断**。

&emsp;&emsp;这条状态主线使用两段相连、但不混为同一个业务问题的案例。第一至第四章使用“服务等级”案例，证明一项新分析能力怎样安全进入运行态；第五章开始切换到已经准备好多条完成分析的“渠道下降”案例，证明可信分析怎样进入归因工作台。两段案例之间通过“新能力先被分析师实际使用，并形成可追溯的完成分析”衔接，不把管理员变更直接说成归因 Evidence。

&emsp;&emsp;学习方法与第一节课保持一致：先看业务问题和页面结果，再回到源码与可运行代码寻找责任边界。代码不是为了堆实现细节，而是让你亲眼看到角色工具、调用顺序、失败出口和测试证据分别能证明什么。

> **目标受众与前置要求**：本课面向已完成前两节可信 NL2SQL 内容，能够阅读 Python、SQL 与 pytest（Python 测试框架）输出的学员。项目 `uv` 环境应可用；PostgreSQL 集成验证需要本机 PG17，真实模型验证需要在本机配置凭据。

> **学完本节你将带走这些能力**：① **角色边界**：区分数据分析师、数据库管理员、模型与确定性代码；② **可信变更**：解释结构化变更意图和七道受控工序；③ **能力验真**：从物理数据库、激活领域版本和分析师运行态证明新能力已经生效；④ **多智能体归因**：解释 ADVOCATE（支持方）、SKEPTIC（质疑方）、JUDGE（裁决方）的职责、工具、固定拓扑与提前收敛条件；⑤ **信任边界**：说明冻结 Evidence、白名单校验、服务端可信装配与公开事件；⑥ **项目表达**：诚实区分单元测试、系统闭环与真实模型多智能体 E2E（端到端）。

> **本课范围与边界**：本课重点为管理员受控变更流程、服务等级跨层变更、归因工作台和项目收尾；不重复前两课可信 NL2SQL 的内部细节。归因工作台不是因果推断、精确贡献分解或自动决策系统。

> **时效与证据边界**：本课统一使用 [HermesAnalytics-第三版](HermesAnalytics-第三版) 目录中的当前源码。该目录不包含可用于重新确认提交号的 Git 元数据，因此不把旧提交号当作当前运行证据。课文中的“实测记录”只证明对应命令和测试范围；确定性 fake agent（确定性替身智能体）系统闭环不能冒充真实模型多智能体 E2E。

> **配套代码验证**：打开 [HermesAnalytics第三节课-治理、归因与责任闭环.ipynb](HermesAnalytics第三节课-治理、归因与责任闭环.ipynb)，可以按课程顺序准备输入、运行探针、观察过程和核对输出。正文负责解释完整知识链，Notebook 负责展示可运行代码与保存结果；每段结果都要结合附近的证据边界理解，不能把局部 `PASS` 外推为完整系统验收。

### 运行 Notebook 前先完成环境准备

&emsp;&emsp;Notebook 与 `HermesAnalytics-第三版` 源码目录保持同级。打开 Notebook 后先运行下面的公共单元：它只确认当前 Conda 环境并加载第三版源码路径，不连接数据库，也不调用真实模型。环境或目录不符合要求时会立即停止，避免后续代码误读其他版本的源码。

In [4]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = (NOTEBOOK_DIR / 'HermesAnalytics-第三版').resolve()
EXPECTED_ENV_NAME = 'hermes-lesson2'
PYTHON_PATHS = (
    PROJECT_ROOT / 'backend' / 'src',
    PROJECT_ROOT / 'database',
    PROJECT_ROOT / 'third_party' / 'hermes-agent',
)

if not PROJECT_ROOT.is_dir():
    raise RuntimeError('没有找到 HermesAnalytics-第三版，请确认 Notebook 与源码目录保持同级')
if Path(sys.prefix).name != EXPECTED_ENV_NAME:
    raise RuntimeError(f'当前环境是 {Path(sys.prefix).name}，请切换到 {EXPECTED_ENV_NAME} 后重新运行')

# reversed + insert(0) 保证最终优先级仍是 backend/src、database、hermes-agent。
for path in reversed(PYTHON_PATHS):
    path_text = str(path)
    if path_text in sys.path:
        sys.path.remove(path_text)
    sys.path.insert(0, path_text)

print('Python 解释器：', Path(sys.executable).name)
print('Python 版本：', sys.version.split()[0])
print('Conda 环境：', Path(sys.prefix).name)
print('源码目录：', 'HermesAnalytics-第三版')
for path in PYTHON_PATHS:
    print('已加载 Python 路径：', path.relative_to(PROJECT_ROOT), '存在：', path.exists())

Python 解释器： python
Python 版本： 3.12.13
Conda 环境： hermes-lesson2
源码目录： HermesAnalytics-第三版
已加载 Python 路径： backend/src 存在： True
已加载 Python 路径： database 存在： True
已加载 Python 路径： third_party/hermes-agent 存在： True


&emsp;&emsp;下面用一张路线表固定全课顺序和两段业务案例的交接位置。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 0-1　课程路线与两段业务案例</font></p>

<div class="courseware-table-wrap">

| 章节 | 这一章要回答的问题 | 主线位置 |
| --- | --- | --- |
| 课程导入 | 本课怎样从管理员变更连续走到归因报告？ | 固定主线、范围与证据边界 |
| 第一章：管理员、分析师与数据对象的职责边界 | 管理员、分析师和三层数据对象分别负责什么？ | 锁定变更对象与权限边界 |
| 第二章：模型如何把需求整理成可检查的变更申请 | 模型能提交什么，为什么不能直接写库？ | 把业务需求变成受限输入 |
| 第三章：管理员变更如何经过检查、预演和确认后安全执行 | 一项变更怎样经过审校、预演、确认与执行？ | 建立安全写入主链 |
| 第四章：执行后如何确认新的分析能力真正生效 | 怎样证明服务等级已经真正可分析？ | 从落库走到分析运行态 |
| 第五章：哪些可信分析可以进入归因工作台 | 哪些可信分析可以冻结为归因 Evidence？ | 从可信读链进入归因链 |
| 第六章：Hermes Agent 如何承担三个归因角色 | 每个角色有什么职责、Prompt（提示词）和专属工具？ | 锁定多智能体能力边界 |
| 第七章：多智能体归因如何按固定顺序运行 | 三个角色先做什么、后做什么，怎样结束？ | 运行开场、反驳与裁决 |
| 第八章：模型输出如何成为可信归因记录 | 模型输出怎样经过校验、装配和状态流转？ | 形成可审计公开记录 |
| 第九章：人类如何审阅归因报告并准确表达项目 | 人类怎样使用报告，怎样向他人准确表达？ | 形成完整项目闭环 |

</div>

> **贯穿本课的两个连续问题**：第一，一项新的“服务等级”分析能力怎样经过管理员受控变更进入分析运行态？第二，分析师完成的“渠道下降”可信分析怎样经过候选复检和启动冻结，成为多智能体归因的 Evidence？

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164215465.png" alt="从可信分析到人类判断的项目闭环" width="85%"></div>

&emsp;&emsp;图解：管理员先通过受约束的变更流程产生可运行的新能力，分析师再使用已治理能力完成可信分析；符合条件的完成态分析在归因启动时冻结为 Evidence，再由受控多角色工作流生成报告，最后交给人类判断。图中表达的是状态衔接，不表示服务等级变更记录会直接进入渠道下降归因。

&emsp;&emsp;事实边界：图中的 Agent 不直连数据库；管理员变更也不直接连到归因 Agent。候选与冻结快照的约束见 `application/attribution/service.py`、`domain/attribution.py` 与 `infrastructure/postgres/attribution_repository.py`。

## <center>第一章：管理员、分析师与数据对象的职责边界</center>

&emsp;&emsp;“我要按服务等级看销售额，为什么分析师问不出来？”先别急着把它归结为模型不会回答。本章按一条可复查的学习路线来拆这个问题：**先分清谁负责什么，再逐层查缺什么，最后读一个只读探针，确认本地规则是否自洽。**这样做的目的不是让你记住更多后台名词，而是让你能判断：一个分析能力卡住时，下一步该找谁、该补哪一层、又有哪些结论暂时不能下。

&emsp;&emsp;贯穿案例仍然是“按服务等级看销售额”。问题可能出在业务事实、治理投影或活动业务语义中的任一层；即使三层都补齐，分析师仍要重新验证这个新维度是否真的能支持当前问题。读完本章，你应能把“问不出来”翻译成一份可检查的能力缺口，而不是把所有责任交给模型。

### 1.1 系统里到底有几个正式角色？

&emsp;&emsp;当前源码中的 `UserRole` 只有两个正式枚举值：`DATA_ANALYST` 与 `DATABASE_ADMIN`。前者从分析入口使用已经治理好的数据能力，后者从管理员入口发起和确认受控变更。Agent 和确定性代码是系统中的两类协作职责，不是另外两个登录角色，也不能与这两个用户身份互相替代。

&emsp;&emsp;`SALES_MANAGER` 只在迁移历史和老库退役兼容分支中保留名字，不是现行角色；即使老库仍有携带该角色的用户行，应用也会把它当作不存在的用户处理。证据见[源码根 `AGENTS.md`](HermesAnalytics-第三版/AGENTS.md)与[`test_a_user_row_carrying_a_retired_role_is_not_a_user_at_all`](HermesAnalytics-第三版/backend/tests/integration/test_postgres_control_repository.py)。

<!-- IMAGE_PROMPT
Use case: scientific-educational
Asset type: 16:9 横向中文技术课件信息图
Teaching objective: 让学习者一眼看懂项目中只有两个正式角色，以及两个角色各自的工作入口和协作关系；两个角色彼此独立，不表达同一用户可以任意切换权限。
Primary request: 从零生成“两个正式角色与工作入口”信息图。左侧蓝色卡为数据分析师，右侧橙色卡为数据库管理员；两张卡之间只展示能力缺口与治理完成后的双向协作，不展示数据库直写、Agent、SQL 或代码实现。
Composition/framing: 顶部居中标题；主体左右两张同等大小的独立角色卡；上方箭头从分析师指向管理员，下方箭头从管理员返回分析师；底部单独放置浅灰色历史兼容说明条，且不与任一正式角色连线。
Text (verbatim): “两个正式角色与工作入口”; “数据分析师”; “DATA_ANALYST”; “使用分析能力”; “分析工作区”; “/analyst/analysis”; “数据库管理员”; “DATABASE_ADMIN”; “治理分析能力”; “管理员入口”; “/admin”; “发现能力缺口”; “发起治理”; “能力可用”; “重新验证”; “SALES_MANAGER：仅历史迁移兼容，不是现行角色”.
Relationships: 上方蓝色箭头严格表达“数据分析师发现能力缺口 → 数据库管理员发起治理”；下方橙色箭头严格表达“管理员完成治理使能力可用 → 数据分析师重新验证”。两条箭头不表示角色或权限互换。
Style: 现代、专业、简洁的企业级数据治理课程插图；白色或极浅灰背景；蓝色代表数据分析师，橙色代表数据库管理员，灰色代表历史兼容；扁平矢量信息图风格，轻微阴影，大字号，路径使用等宽字体样式。
Constraints: 所有中文、英文枚举和路径必须逐字准确、各出现一次；两张角色卡使用独立人物；历史说明条必须与主体隔离；不得增加提示词之外的文字、角标、Logo、水印或装饰性英文。
Avoid: 同一人物分叉、第三个现行角色、角色任意切换、数据库直写、Hermes Agent、SQL、代码流程、额外箭头、额外文字、乱码、3D、写实场景、卡通吉祥物。
-->

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164249390.png" alt="两个正式角色、工作入口与能力治理协作关系" width="85%"></div>

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 1-1　常见误解、真实机制与判断边界</font></p>

<div class="courseware-table-wrap">

| 容易误解成什么 | 当前机制 | 判断边界 |
| --- | --- | --- |
| 四类职责等于四个用户角色 | 应用层正式角色只有 `DATA_ANALYST` 与 `DATABASE_ADMIN`；Agent、确定性代码承担协作职责 | 看到“四方”时，先问它说的是职责，还是 `UserRole` 中的登录身份 |
| 管理员确认等于逐行审 SQL | 管理员检查业务目标、影响与预演并作人工确认；结构审校由 `review_structure` 在本地检查申请形状 | 管理员确认不能替代系统检查；结构正确也不等于后续真实前置条件已经满足 |
| Agent 会直接执行任意写库 | 当前变更步骤模型不含 SQL；结构审校也明确是纯本地检查 | 通过 HermesAnalytics 应用入口，任何角色都不能绕过受控链直接执行任意写库；这句话不外推为数据库世界中所有工具、账户都不可能写库 |
| Policy 放行就说明对象已经可用 | `allows_table` / `allows_view` 只判断对象名是否落在构造出的 Policy 范围内 | 放行只让对象进入后续检查，不证明对象已存在、视图已发布、数据正确，或分析已经成功 |
| Skills 是第四个数据层 | Skills 约束 Agent 怎样收集事实、怎样组织申请；三层对象仍是业务、治理、语义对象 | Skill 不会自动成为表、视图或活动领域版本，也不能补齐其中任意一层 |

</div>

#### 两个角色怎样与 Agent、确定性代码协作？

&emsp;&emsp;两个正式角色要完成一次能力补齐，需要与 Agent 和确定性代码形成四类职责闭环：

- **数据分析师负责使用已经治理好的能力**：在规定的分析入口提问、阅读结果、继续分析，并在能力补齐后重新验证问题与结果。
- **数据库管理员负责治理能力**：确认能力缺口，检查业务影响和预演结果，并作最终人工确认；这属于业务审核和风险决策。
- **Agent 负责整理申请**：读取受限事实与 Skills，把自然语言需求整理为系统可检查的结构化变更意图；当前 `ChangeStep` 的模型说明明确它不含 SQL。
- **确定性代码负责可重复的技术检查与执行**：完成结构审校、编译、自检、预演、冻结和受限执行，并把失败停在明确边界内。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 1-2　角色与系统职责的白话解释</font></p>

<div class="courseware-table-wrap">

| 参与者 | 主要负责 | 不负责 |
| --- | --- | --- |
| 数据分析师 `DATA_ANALYST` | 使用已经治理好的数据能力完成分析 | 不直接改业务表、治理视图或领域语义 |
| 数据库管理员 `DATABASE_ADMIN` | 发起、审核和确认数据能力变更 | 不绕过系统随意执行 SQL，也不亲自完成每一行技术代码 |
| Agent | 把自然语言需求整理为可检查的变更意图 | 不直接连接数据库执行任意写操作，也不替代管理员作最终人工确认 |
| 确定性代码 | 结构审校、编译、自检、预演、冻结和受限执行 | 不自行判断业务目标、风险是否可接受，也不替代管理员作最终人工确认 |

</div>

&emsp;&emsp;生活中可以把它类比为餐厅新增一道菜：分析师像点单的顾客，管理员像确认菜单和风险的负责人，Agent 像把口头需求写成标准工单的前台，确定性代码像按配方和卫生规则逐项核验的后厨流程。这个类比只帮助理解“职责不能互相跳过”；它**不对应**真实数据库权限、SQL 编译、部署 Policy 的具体实现，也不表示管理员或 Agent 在系统外没有别的工具。

&emsp;&emsp;这里还会用到两个简称：**可信读链**是分析师通过规定入口读取已经治理和发布的数据；**受控写链**是管理员通过“提出变更、检查、预演、人工确认、执行、验证”的受约束流程推动数据能力变化。管理员是变更的发起者、审核者和确认者，Agent 帮助理解需求，系统代码负责技术检查与执行；这不表示管理员拥有数据库超级权限，也不表示全部工作都由管理员手工完成。

> **“管理员审核”与“系统审校”不是同一个动作**：管理员主要检查业务目标、影响范围、预演结果与风险，并决定是否发起最终执行；源码中的 `review/审校` 则由确定性代码完成，先检查申请结构、步骤顺序和已声明的 Skill。结构审校通过后，真实对象、数据与其他前置条件仍需在后续环节核对。管理员不需要手工逐行审核编译器生成的每条 SQL，也不能绕过系统审校直接放行。

```text
分析师发现“服务等级”问不出来
        ↓
管理员确认需要补充这项数据能力
        ↓
Agent 把自然语言需求整理成系统可检查的变更申请
        ↓
系统代码审校、编译、自检、预演和冻结
        ↓
管理员最终人工确认
        ↓
受限执行并验证结果
        ↓
分析师重新提问并验证
```

&emsp;&emsp;下图中的“四方”是四类职责，不是四个应用角色。它总结的是从发现缺口到能力重新可用的协作闭环；当前正式用户角色仍然只有 `DATA_ANALYST` 与 `DATABASE_ADMIN`。

<!-- IMAGE_PROMPT
Use case: scientific-educational
Asset type: 16:9 横向中文课件职责分工信息图
Primary request: 为 HermesAnalytics 数据分析课程绘制“四方职责与协作边界”。用四张并列职责卡分别展示 DATA_ANALYST 数据分析师、DATABASE_ADMIN 数据库管理员、Hermes Agent、确定性代码，并用底部闭环展示一项新分析能力从发现缺口到重新可用的完整过程。必须明确区分管理员的业务检查与最终人工确认、确定性代码的结构审校与受限执行；Agent 不能直接写数据库。
Composition: 顶部居中标题“四方职责与协作边界”；中部从左到右排列蓝色、橙色、靛蓝色、蓝绿色四张圆角职责卡；底部使用七个等距圆角节点和箭头形成闭环；数据库图标仅位于确定性代码下方，并且只有确定性代码通过蓝绿色实线连接数据库；Hermes Agent 到数据库使用红色虚线、禁止符号和“禁止直接写库”。
Text (verbatim): “四方职责与协作边界”; “DATA_ANALYST 数据分析师”; “提出业务问题”; “使用已有能力”; “验证分析结果”; “DATABASE_ADMIN 数据库管理员”; “确认能力缺口”; “检查影响与预演”; “最终人工确认”; “Hermes Agent”; “读取受限事实与 Skills”; “整理 ChangeIntent”; “信息不足时澄清”; “确定性代码”; “审校、编译与冻结”; “dry-run 与受限执行”; “终态回读与审计”; “发现缺口”; “发起治理”; “整理申请”; “检查与预演”; “人工确认”; “受限执行”; “能力可用”; “数据库”; “禁止直接写库”.
Relationships: 底部流程严格按照“发现缺口 → 发起治理 → 整理申请 → 检查与预演 → 人工确认 → 受限执行 → 能力可用”排列，再由“能力可用”回到下一轮“发现缺口”；数据库是执行目标，不是发现缺口的主体。
Style: 专业、简洁、扁平化的科学教育信息图；浅色背景、深色高对比中文、充足留白；分析师蓝色、管理员橙色、Agent 靛蓝色、确定性代码和合法执行路径蓝绿色，红色只用于禁止路径。
Constraints: 所有简体中文必须逐字清晰、水平排版、字号足以大屏展示；不得省略、改写、重复或生成乱码；管理员卡只表达业务影响检查和人工授权，不表达手工逐行审 SQL；Agent 没有数据库写工具；分析师和管理员都不能绕过系统代码直接修改数据库。
Avoid: 3D、写实人物、卡通吉祥物、密集段落、竖排中文、低对比渐变、Logo、水印、额外英文、SQL 代码、额外角色、SALES_MANAGER、Agent 直连数据库、管理员直连数据库、把人工审核与系统审校混为一谈。
-->

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164226570.png" alt="数据分析师、数据库管理员、Hermes Agent 与确定性代码的职责及协作边界" width="85%"></div>

#### 分析能力缺失时，到底缺哪个对象？

&emsp;&emsp;管理员改变的不是一个抽象“后台”，而是三层有明确归属的数据与语义对象。下表的 `ecommerce_core` 和 `analytics` 是**当前教学部署配置中**用于探针的 schema 名，不是程序硬编码，也不是所有部署唯一允许使用的 schema。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 1-3　三类对象分别解决什么问题</font></p>

<div class="courseware-table-wrap">

| 层 | 真正对象 | 用白话理解 |
| --- | --- | --- |
| 业务层 | 当前教学部署配置中的 `ecommerce_core` 表、列、索引和数据 | 数据库里是否真的有“服务等级”这个字段和值 |
| 治理层 | 当前教学部署配置中的 `analytics` 治理视图 | 分析师使用的规定入口是否能看到这个字段 |
| 语义层 | 活动领域版本中的维度、指标、公式 | 系统是否知道“服务等级”可以用来分组，是否知道“销售额”该怎样计算 |

</div>

&emsp;&emsp;语义层不能理解成“目录里只要有一个文件就行”。`DomainCatalogService.get_active()` 会先读取当前活动版本，再取回该版本的 Catalog；因此需要检查的是**活动领域版本**是否给出可用语义，而不是任意草稿是否存在。

&emsp;&emsp;Skills 不属于业务层、治理层或语义层。它是 Agent 处理变更任务时使用的操作说明和经验规则，告诉 Agent 面对加字段、改视图或增加维度时怎样收集事实、怎样组织申请。Skills 构成独立的方法治理面；Skill 本身不会直接变成业务表、治理视图或活动领域版本，也不能替代其中任意一层。

<!-- IMAGE_PROMPT
Use case: scientific-educational
Asset type: 16:9 横向中文技术课件信息图
Teaching objective: 让学员一眼理解业务层、治理层、语义层分别代表什么，以及数据分析师实际只读治理视图、不能直接读取业务表；用“服务等级”贯穿三层，解释它怎样成为可用分析维度。
Primary request: 生成简洁、准确的“三层如何共同支撑分析”信息图。不要复刻旧图，不展示 Skills、仓库流程、认可约定、PENDING 提案或管理员七阶段流程。
Reading order: 先从左侧由下到上阅读业务层、治理层、语义层，再看右侧 DATA_ANALYST 数据分析师。业务事实由业务层通过关联、筛选、投影进入治理层；语义层指导分析规划；分析师的受控查询只读取治理层。
Relationships: 业务层通过向上实线箭头连接治理层；治理层通过向右实线箭头连接分析师；语义层通过向右下实线箭头连接“分析规划”；分析师到业务层使用带禁止符号的红色虚线，表达“禁止直接读取业务表”。不得画治理层数据流入语义层。
Subject/nodes: 左侧由下到上三张宽大的水平圆角卡，分别为蓝色业务层、紫色治理层和绿色语义层；右侧为深蓝边框分析师卡，并显示分析规划与最终业务问题。
Text (verbatim): “三层如何共同支撑分析”; “语义层”; “激活领域版本”; “指标、维度、公式”; “服务等级登记为维度”; “治理层”; “analytics”; “治理视图”; “服务等级字段已开放”; “业务层”; “ecommerce_core”; “业务表、字段、数据”; “orders 中存在服务等级和值”; “关联、筛选、投影”; “分析规划”; “DATA_ANALYST”; “数据分析师”; “按服务等级看销售额”; “分析查询只读治理视图”; “禁止直接读取业务表”; “三层共同满足，服务等级才是可用分析维度”.
Style: 专业、简洁、扁平化科学教育信息图；极浅灰白背景；网格对齐；大字号；业务层蓝色、治理层紫色、语义层绿色、分析师深蓝色；红色只用于禁止路径。
Constraints: 严格保持业务层保存事实、治理层提供可读视图、语义层提供指标维度与公式；分析师只能从治理层获得查询结果，并由语义层指导规划；中文和技术标识符必须逐字准确，无错字、缺字、乱码、重复或额外文字。
Avoid: Skills 方法治理面、仓库流程区、认可约定区、现场拼装、PENDING 提案、SALES_MANAGER、DATABASE_ADMIN、Hermes Agent、七阶段流程、SQL、数据库直连、额外角色、额外箭头、Logo、水印、3D、写实人物、密集小字、竖排文字。
Precise final edit: 若语义层绿色箭头下方出现模糊蓝色乱码或额外文字，只删除该处并保持干净留白；标题、三层卡片、箭头、分析师卡、禁止路径、底部结论及其他文字全部保持不变。
-->

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164325520.png" alt="业务层、治理层、语义层共同支撑分析及分析师读取边界" width="85%"></div>

&emsp;&emsp;回到贯穿案例：服务等级即使先出现在业务表里，只要没有治理投影和活动语义，分析师仍然答不出“我要按服务等级看销售额”。下面的矩阵把“缺哪一层”具体化；它只针对**新增服务等级分析维度**这个案例，不外推为所有分析问题都只能按这三格排查。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 1-4　服务等级案例的逐层失败矩阵</font></p>

<div class="courseware-table-wrap">

| 当前发现 | 这个案例说明什么 | 此时不能下的结论 |
| --- | --- | --- |
| 业务事实缺失 | 还没有可供分析的“服务等级”字段或值 | 不能只靠加语义定义就让结果出现 |
| 业务事实存在，但治理投影缺失 | 字段存在，却未成为分析入口可读取的治理对象 | 不能把“库里有列”当成分析师已经能使用 |
| 业务、治理都具备，但活动语义缺失 | 系统尚未在活动领域版本中把它组织成可规划的维度与指标关系 | 不能只凭字段名推断系统会如何分组或计算 |
| 三层齐备 | 该案例的新增维度具备继续分析的对象基础 | 仍需分析师重新验证当前问题、结果和业务解释；不等于所有分析自动正确 |

</div>

&emsp;&emsp;遇到类似的能力缺口时，可以依次判断：**业务事实是否存在，治理视图是否开放，活动语义是否理解。**任意一层缺失，分析师都不能把“服务等级”当作可信分析维度；三层齐备后也只是具备对象基础，仍要由分析师重新提问，核对结果和业务解释。

&emsp;&emsp;源码锚点：[角色枚举](HermesAnalytics-第三版/backend/src/hermes_analytics/domain/identity.py)、[鉴权依赖](HermesAnalytics-第三版/backend/src/hermes_analytics/api/dependencies/auth.py)、[部署安全边界](HermesAnalytics-第三版/backend/src/hermes_analytics/admin_change/schema_policy.py)、[变更意图模型 `ChangeStep` / `ChangeIntent`](HermesAnalytics-第三版/backend/src/hermes_analytics/admin_change/models.py)、[结构审校](HermesAnalytics-第三版/backend/src/hermes_analytics/admin_change/review/structure.py)、[活动领域目录](HermesAnalytics-第三版/backend/src/hermes_analytics/nl2sql/catalog/service.py)。

### 1.2 Policy 探针研究的是什么？

&emsp;&emsp;这一段代码研究的不是“数据库能不能改成功”，而是管理员受控变更链路的第一道对象范围边界。`AdminSchemaPolicy` 可以理解为**对象范围白名单**：它只回答“哪些对象名有资格进入后续变更检查？”

&emsp;&emsp;这里的 Policy 不是 PostgreSQL RLS 中的 `CREATE POLICY`，不是给每张表安装的数据库 Policy，也不是数据库管理员的完整权限表。它不会检查数据库对象是否真实存在，更不能证明变更已经成功。

&emsp;&emsp;角色与对象范围是两个不同问题：

- `UserRole` 回答“谁可以进入哪个应用入口”；
- `AdminSchemaPolicy` 回答“管理员受控变更链路最多可以把哪些对象作为候选目标”；
- Policy 放行不等于管理员已经获得直接写数据库的权限。

&emsp;&emsp;手工构造 Policy 时，三个参数分别限制不同对象范围：`table_schemas` 允许哪些 schema 作为业务表目标，`view_schemas` 允许哪些 schema 作为治理视图目标，`controlled_system_relations` 只精确放行少数 `system.object` 内部关系，而不是开放整个 `system` schema。

&emsp;&emsp;下面的探针只观察四件事：正式角色枚举；业务表对象名是否被当前手工构造的 Policy 放行；治理视图对象名是否被放行；把 `system` 当作普通业务表范围时是否立即拒绝。它不连接数据库，也不读取已启动服务的真实部署 Policy。

&emsp;&emsp;因此，探针不能证明对象真实存在、当前部署采用了这份配置、数据库账号已经授权、治理视图已经发布、变更已经执行成功，或分析师已经能够完成端到端分析。对象被 Policy 放行后，仍需经过：**结构审校 → 编译 → 自检 → 预演 → 人工确认 → 受限执行器 → 数据库账户权限 → 结果验证**。Policy 是第一道对象范围边界，不是完整安全体系。

&emsp;&emsp;运行目录：`HermesAnalytics-第三版/backend`。确认项目的 `uv` 环境可用后运行下面的探针，重点观察正式角色、业务表范围、治理视图范围和受保护 schema 的拒绝分支。代码只在内存中构造 Policy，不连接数据库，也不读取已启动服务的部署配置。

In [2]:
"""读取正式角色与部署安全边界，并把判断过程打印出来。"""
from hermes_analytics.admin_change.schema_policy import AdminSchemaPolicy
from hermes_analytics.domain.identity import UserRole

# 角色枚举是应用层的正式角色真源，先直接展示源码当前值。
roles = sorted(role.value for role in UserRole)
print('应用层正式角色：', roles)

# 从部署配置创建对象范围白名单；这里只判断目标是否在允许范围内，
# 不查询数据库对象是否存在，也不代表变更已经执行成功。
policy = AdminSchemaPolicy.from_deployment_config(
    table_schemas='ecommerce_core',                             # 业务表目标只允许位于 ecommerce_core。
    view_schemas='analytics',                                   # 治理视图目标只允许位于 analytics。
    controlled_system_relations=('system.change_requests',),    # system 不整体开放，只精确放行该关系。
)
# 分别打印业务表与治理视图的判断结果，便于观察两个入口的边界。
objects = (
    ('业务表 ecommerce_core.orders', policy.allows_table('ecommerce_core.orders')),
    ('治理视图 analytics.v_paid_order_items', policy.allows_view('analytics.v_paid_order_items')),
)
for object_name, allowed in objects:
    print(f'{object_name} 是否被部署 Policy 放行：{allowed}')

# 再走一遍反向分支：system 是受保护 schema，不能伪装成业务表范围。
try:
    AdminSchemaPolicy.from_deployment_config(
        table_schemas='system',                                     # 故意写入非法范围，演示受保护 schema 被拒绝。
        view_schemas='analytics',                                   # 保持合法，便于锁定 table_schemas 的错误。
        controlled_system_relations=('system.change_requests',),    # 保持合法，不干扰本次反向分支。
    )
except ValueError as error:
    print('尝试把 system 配成业务表范围时，系统拒绝：', str(error))
else:
    print('system 被意外接受，需要检查部署安全配置。')

应用层正式角色： ['DATABASE_ADMIN', 'DATA_ANALYST']
业务表 ecommerce_core.orders 是否被部署 Policy 放行：True
治理视图 analytics.v_paid_order_items 是否被部署 Policy 放行：True
尝试把 system 配成业务表范围时，系统拒绝： 部署安全策略把受保护 schema 当成业务范围：('system',)


&emsp;&emsp;输出需要逐项理解：

1. **角色列表**：只说明 `UserRole` 当前枚举出了哪些应用层正式角色，不证明浏览器入口或鉴权链已经实际跑通。
2. **业务表判断为 `True`**：只说明 `ecommerce_core.orders` 的 schema 和标识符格式落在这次手工构造的 Policy 业务表范围内。
3. **治理视图判断为 `True`**：只说明 `analytics.v_paid_order_items` 的 schema 和标识符格式落在这次手工构造的 Policy 治理视图范围内。
4. **`system` 被拒绝**：说明受保护的 `system` schema 不能整体伪装成普通业务表范围；这不表示所有内部关系都永远不能操作，少数内部关系仍可通过 `controlled_system_relations` 精确放行。
5. **出现“`system` 被意外接受”**：说明反向分支没有按预期拒绝，应停止继续解释结果并检查部署安全配置。

&emsp;&emsp;两个 `True` 都不表示对象真实存在，不表示已启动部署采用这份配置，不表示数据库账号有权限，也不表示变更已经成功。它们只证明本地枚举与这次手工构造的对象范围规则在当前输入下给出了这些判断。

### 1.3 学习检查

1. 系统中有几个正式用户角色？为什么 Agent 和确定性代码不能算作另外两个登录角色？
2. 分析师发现“服务等级”无法分析后，管理员、Agent 和确定性代码分别接手什么职责？管理员审核与系统审校有什么不同？
3. “服务等级”成为可信分析维度需要检查哪三层？为什么 Skills 不能算作第四个数据层？
4. Policy 探针中的两个 `True` 与 `system` 拒绝分别能证明什么，又不能证明什么？

<details><summary>参考要点</summary>

1. 正式角色只有 `DATA_ANALYST` 与 `DATABASE_ADMIN`；Agent 和确定性代码承担系统职责，不是应用登录身份；`SALES_MANAGER` 只属于迁移历史和退役兼容。
2. 分析师使用并重新验证能力；管理员确认缺口、影响与预演并作最终人工确认；Agent 整理结构化变更意图；确定性代码负责结构审校、编译、自检、预演、冻结和受限执行。管理员审核业务目标与风险，系统审校申请结构和执行前置条件，两者不能互相替代。
3. 需要检查业务事实、治理投影和活动语义；Skills 是 Agent 使用的操作说明与经验规则，不会直接成为业务表、治理视图或活动领域版本。
4. 两个 `True` 只表示对象名落在手工 Policy 的 schema 与标识符格式范围内；`system` 拒绝只说明受保护 schema 不能整体开放。它们都不证明对象存在、部署配置、账号权限、执行成功或端到端分析可用。

</details>

&emsp;&emsp;最后用一句判断法收束：**角色回答“谁负责”，三层回答“缺什么”，Policy 探针回答“对象名能否进入后续检查”。**三者一起能帮助你定位下一步，但都不能单独替代真实运行、数据库验真和分析师复核。

## <center>第二章：模型如何把需求整理成可检查的变更申请</center>

&emsp;&emsp;管理员提出：“给订单表增加服务等级字段。”这句话表达了业务目标，但还没有说明稳定的机器动作、目标对象、字段类型、依赖关系和风险边界。模型在这里负责把自然语言整理成一份系统可以继续检查的申请，而不是把一句话直接变成数据库写入。

&emsp;&emsp;需要先守住一条边界：`ChangeIntent` 是“变更申请”，不是 SQL，也不是执行命令。即使申请字段完全合规，服务等级字段此时仍未创建，数据库也没有发生变化。

### 2.1 管理员说完一句话，模型允许交出什么？

&emsp;&emsp;管理员意图 Agent 只能读取受限事实和操作说明，再从三种允许结果中选择一种：

1. **信息不足时提交澄清**：把缺少的事实重新问给管理员，不能靠猜测补全。
2. **确认无需变化时提交无需变更**：明确结束本次申请，不为了“必须产出”而制造无意义变更。
3. **信息足够时提交 `ChangeIntent`**：把业务目标整理成字段明确、可由确定性代码检查的结构化申请。

&emsp;&emsp;模型没有数据库执行工具，也不负责生成最终 SQL。自然语言理解适合判断“管理员想做什么”，但数据库写入需要稳定、可重复的规则；如果模型同时拥有自由 SQL 和写库权限，一次对象选择或风险判断偏差就可能直接变成数据事故。

&emsp;&emsp;把这份申请放回完整流程，可以看到它只位于第二阶段：

```text
管理员提出业务需求
        ↓
模型读取只读事实，提交澄清、无需变更或 ChangeIntent
        ↓
确定性代码审校、编译、Policy 检查、冻结和 dry-run（预演）
        ↓
管理员检查预演并作最终人工确认
        ↓
受限执行、终态回读与审计
```

&emsp;&emsp;下图把这条主链压缩为五个阶段。注意红色虚线：模型到数据库的直接写入路径被明确截断；合法执行只能从确定性受限执行器进入数据库。

<!-- IMAGE_PROMPT
Use case: scientific-educational
Asset type: 16:9 横向中文技术课件流程信息图
Teaching objective: 让学员沿一条清楚主链理解管理员变更从业务需求、模型整理、确定性检查、人工确认，到受限执行与审计的完整职责边界。
Primary request: 从零生成“管理员变更如何安全落地”五阶段旅程图，不使用旧式四泳道表格。五个编号阶段沿单一主方向从左到右排列，角色、动作、澄清分支、人工 Gate 和禁止路径一眼可分。
Reading order: 顶部标题 → 1 至 5 五个阶段 → 模型的澄清返回分支 → 底部红色禁止路径 → 底部总结句。
Relationships: 主链严格为“1 管理员提出需求 → 2 模型整理申请 → 3 确定性代码检查 → 4 管理员人工确认 → 5 受限执行与审计”。阶段 2 先读取只读事实，再由“信息足够？”分为“不足：澄清”返回管理员和“足够：ChangeIntent”进入阶段 3。阶段 3 严格展示“审校 → 编译 → Policy → 冻结 → dry-run”。阶段 4 同时满足“预演通过”和“人工确认”后才放行。阶段 5 严格展示“受限执行 → 数据库 → 终态回读 → 审计”。
Prohibited path: 从阶段 2 的模型卡片向阶段 5 的唯一数据库节点绘制红色虚线，并在到达数据库前用禁止符号截断，文字为“禁止模型直接写库”。不得画管理员直接连接数据库。
Text (verbatim): “管理员变更如何安全落地”; “1 管理员提出需求”; “业务需求”; “2 模型整理申请”; “读取只读事实”; “信息足够？”; “不足：澄清”; “足够：ChangeIntent”; “3 确定性代码检查”; “审校”; “编译”; “Policy”; “冻结”; “dry-run”; “4 管理员人工确认”; “预演通过”; “人工确认”; “放行”; “5 受限执行与审计”; “受限执行”; “数据库”; “终态回读”; “审计”; “禁止模型直接写库”; “模型整理申请｜代码检查执行｜管理员最终确认”.
Style: 专业、现代、简洁的扁平矢量教学信息图；浅暖白或极浅灰背景；模型与输入使用蓝色，确定性代码和合法执行使用蓝绿色，管理员与人工 Gate 使用橙色，红色只用于禁止路径；人物成熟、亲和但不幼稚。
Constraints: 五个大阶段编号必须严格为 1、2、3、4、5；主链只有一条合法路径；dry-run 在人工确认之前，人工确认在受限执行之前；全图只出现一个数据库节点；模型只允许提交 ChangeIntent、澄清或无需变更，不显示 SQL、执行按钮或数据库写工具；背景覆盖完整画布，无透明通道或黑色空洞。
Avoid: 旧式泳道网格、重复数据库、模型直连数据库、管理员直连数据库、模型编译或执行 SQL、管理员手写 SQL、绕过人工 Gate、交叉长折线、额外角色、额外箭头、额外文字、Logo、水印、3D、照片写实、卡通吉祥物、低对比度和密集小字。
-->

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164325549.png" alt="管理员需求、模型整理、确定性检查、人工确认与受限执行审计的五阶段旅程" width="85%"></div>

#### 一份申请里，哪些内容给人看，哪些内容给代码看？

&emsp;&emsp;管理员的原话是业务输入，`ChangeIntentInput` 才是 Agent 整理后的整份申请。申请中既有方便人理解的标题和摘要，也有校验器、编译器和测试共同使用的稳定机器字段，两者不能互相替代。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 2-1　结构化申请各字段回答什么问题</font></p>

<div class="courseware-table-wrap">

| 字段或对象 | 回答的问题 | 判断边界 |
| --- | --- | --- |
| 管理员自然语言输入 | 业务上想解决什么 | 是输入，不是结构化申请，更不是 SQL |
| `intent_schema_version`、`title`、`steps` | 使用哪版协议、整份申请叫什么、包含哪些步骤 | `title` 给人看，不能替代步骤中的机器字段 |
| `order`、`depends_on` | 当前步骤排第几、依赖哪些前置步骤 | 让多步骤申请具有稳定顺序和依赖关系 |
| `category`、`summary`、`skill_id` | 属于哪类变化、怎样向人说明、依据哪份操作说明 | `summary` 是人类说明；`skill_id` 是依据记录，不是 Skill 调用 |
| `values` | 当前步骤真正准备操作哪个对象、采用什么机器动作和参数 | 确定性代码实际检查这些字段 |
| `irreversible` | 当前动作是否不可逆 | 不属于 Agent 工具输入，由服务端规则判断 |

</div>

&emsp;&emsp;以“新增服务等级字段”为例，`title='新增服务等级'` 和 `summary='新增服务等级列'` 帮助管理员阅读；`action='add_column'`、`table='ecommerce_core.orders'`、`column='service_level'`、`data_type='text'` 和 `nullable=True` 才是代码继续检查的具体参数。界面可以显示中文“加列”，机器协议仍必须使用稳定值 `add_column`。

&emsp;&emsp;四类变化对应不同对象，不能看到“新增”两个字就一律当成加字段：

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 2-2　四类变更分别作用于什么对象</font></p>

<div class="courseware-table-wrap">

| 机器类别 | 白话解释 | 服务等级案例中的例子 |
| --- | --- | --- |
| `table_structure` | 修改表、列或索引 | 给订单表增加服务等级字段 |
| `governance_view` | 修改分析师使用的治理视图 | 把服务等级投影给分析师 |
| `semantics` | 修改维度、指标或领域语义 | 把服务等级登记为可分组维度 |
| `business_data` | 修改或补充业务数据 | 为已有订单回填服务等级值 |

</div>

#### `read_skill` 与 `skill_id` 到底有什么区别？

&emsp;&emsp;管理员 Agent 不是默认继承并自动执行全部 Skills。当前机制是**目录提示 → 按需读取 → 留下读取记录 → 提交时核对**：

1. 系统只向 Agent 提供明确工具，其中包括 `list_skills` 和 `read_skill`。
2. Agent 先看到 Skills 目录，再根据当前需求和只读结构事实选择相关说明。
3. 确认需要新增字段时，Agent 调用 `read_skill('add-column')` 读取完整说明；成功后，系统把它记入本轮 `skills_read`。
4. Agent 提交申请时，在对应步骤写入 `skill_id='add-column'`，留下“这一步依据哪份说明形成”的记录。
5. 如果申请声明了本轮没有读过的 Skill，完整提交边界会以 `CHG-REVIEW-SKILL-UNREAD` 拒绝。

&emsp;&emsp;因此，`read_skill` 是实际读取操作，`skill_id` 是申请里的依据标识和审计线索。`skill_id` 本身不会读取 Skill，也不会执行加字段。Skill 像填表说明，`values` 才是这次真正填入的表名、列名、类型等答案。

&emsp;&emsp;实际选择取决于当前结构事实：来源字段已经在治理视图中时，新增指标可能只需要 `create-metric`；字段尚未开放时，可能先需要 `change-governance-view`；底层连原始字段都没有时，才可能形成 `add-column → change-governance-view → create-metric` 的多步骤申请。

#### 为什么风险不能由模型自己填写？

&emsp;&emsp;`ChangeStepInput` 没有 `irreversible` 字段。模型负责说明“准备做什么”，服务端再根据动作类型和确定性规则判断“风险有多高”，真正的 SQL 或领域包补丁也由确定性编译器生成。不能让申请方通过自报“没有风险”降低自己的检查门槛。

&emsp;&emsp;如果 Agent 把中文展示词“加列”直接放进机器字段 `action`，工具契约会指出 `action` 不合法；合法机器值应是 `add_column`。这类拒绝只表示申请字段没有通过检查，不会对数据库产生影响。

&emsp;&emsp;源码锚点：[内部模型](HermesAnalytics-第三版/backend/src/hermes_analytics/admin_change/models.py)、[管理员工具边界](HermesAnalytics-第三版/backend/src/hermes_analytics/hermes_adapter/admin/tools.py)、[提交转换与临时内部对象](HermesAnalytics-第三版/backend/src/hermes_analytics/application/admin_intent/service.py)、[不可逆性推导](HermesAnalytics-第三版/backend/src/hermes_analytics/application/admin_changes/service.py)。

### 2.2 如何用代码看见申请通过和拒绝？

> **管理员输入**
>
> 给订单表增加服务等级字段。

&emsp;&emsp;下面的探针把这句话与结构化申请分开：代码直接构造 `ChangeIntentInput`，用于观察工具契约怎样接受合法机器值、拒绝非法机器值。它模拟 Agent 整理需求，但不调用真实大模型，不调用完整提交函数，也不连接数据库。

&emsp;&emsp;运行目录：`HermesAnalytics-第三版/backend`。运行时先观察管理员原话，再比较合法 `add_column` 与非法中文动作“加列”产生的两条分支；这段探针只检查结构化工具输入，不调用真实模型、完整提交服务或数据库。

In [3]:
"""把管理员的一句话整理成结构化申请，并展示通过与拒绝的完整过程。"""
import json

from hermes_analytics.hermes_adapter.admin.tools import ChangeIntentInput, ChangeStepInput

ADMIN_REQUEST = '给订单表增加服务等级字段'


def make_input(action: str) -> ChangeIntentInput:
    """模拟 Agent 使用指定机器动作整理同一条管理员需求。"""
    return ChangeIntentInput(
        intent_schema_version=2,                                # 使用第 2 版变更申请协议。
        title='新增服务等级',                                   # 给管理员看的整份申请标题。
        steps=[                                                 # 一份申请可以包含一个或多个变更步骤。
            ChangeStepInput(
                order=1,                                       # 当前步骤序号，从 1 开始。
                depends_on=[],                                 # 空列表表示不依赖其他步骤。
                category='table_structure',                    # 这是一项业务表结构变更。
                summary='新增服务等级列',                      # 给管理员看的步骤摘要。
                skill_id='add-column',                         # 声明本步骤依据 add-column Skill。
                values={                                       # 供确定性代码检查的具体机器字段。
                    'action': action,                           # 机器动作；合法值是 add_column。
                    'table': 'ecommerce_core.orders',           # 要修改的目标业务表。
                    'column': 'service_level',                  # 要新增的物理字段名。
                    'data_type': 'text',                        # 新字段使用文本类型。
                    'nullable': True,                           # 新字段暂时允许为空。
                },
            )
        ],
    )


print('【输入】管理员提出自然语言需求')
print('管理员：', ADMIN_REQUEST)
print('说明：这是业务输入。下面的代码模拟 Agent 整理需求，不调用真实大模型。')

accepted = make_input('add_column')
accepted_issues = accepted.step_issues()
accepted_payload = accepted.model_dump(mode='json')
print('\n【步骤 1】Agent 把自然语言整理成结构化申请')
print(json.dumps(accepted_payload, ensure_ascii=False, indent=2))
print('说明：上面的 JSON 是变更申请，不是 SQL，也不代表数据库已经发生变化。')
print('说明：skill_id 是申请中的依据标识；本探针不验证 Agent 是否真实读取过该 Skill。')

print('\n【步骤 2】确定性代码检查合法机器动作 add_column')
print('字段契约问题数量：', len(accepted_issues))
if accepted_issues:
    for issue in accepted_issues:
        print(f'第 {issue.step_order} 步，字段 {issue.field}：{issue.message}')
else:
    print('校验结果：未发现字段契约问题，可以进入后续审查。')
print('说明：问题数量为 0 只表示当前申请格式合规，不代表审批、编译或执行成功。')

has_irreversible = 'irreversible' in accepted_payload['steps'][0]
print('\n【步骤 3】检查风险字段由谁决定')
print('Agent 的工具输入是否包含 irreversible：', has_irreversible)
print('说明：False 表示 Agent 不能自行声明风险；它不表示服务端已判定本次操作可逆。')

issues = make_input('加列').step_issues()
print('\n【步骤 4】把中文展示词“加列”误填为机器动作')
print('页面展示词：加列；机器协议值：add_column。')
for issue in issues:
    print(f'第 {issue.step_order} 步，字段 {issue.field}：{issue.message}')
print('说明：系统拒绝的是不符合机器协议的 action 值，而不是拒绝中文界面。')

print('\n【本章边界】')
print('本探针证明：自然语言需求可以被整理成可检查的申请，非法字段值会被准确定位。')
print('本探针不证明：真实模型已经理解需求、Skill 已真实读取、SQL 已生成或数据库已修改。')

【输入】管理员提出自然语言需求
管理员： 给订单表增加服务等级字段
说明：这是业务输入。下面的代码模拟 Agent 整理需求，不调用真实大模型。

【步骤 1】Agent 把自然语言整理成结构化申请
{
  "intent_schema_version": 2,
  "title": "新增服务等级",
  "steps": [
    {
      "order": 1,
      "depends_on": [],
      "category": "table_structure",
      "summary": "新增服务等级列",
      "skill_id": "add-column",
      "values": {
        "action": "add_column",
        "table": "ecommerce_core.orders",
        "column": "service_level",
        "data_type": "text",
        "nullable": true
      }
    }
  ]
}
说明：上面的 JSON 是变更申请，不是 SQL，也不代表数据库已经发生变化。
说明：skill_id 是申请中的依据标识；本探针不验证 Agent 是否真实读取过该 Skill。

【步骤 2】确定性代码检查合法机器动作 add_column
字段契约问题数量： 0
校验结果：未发现字段契约问题，可以进入后续审查。
说明：问题数量为 0 只表示当前申请格式合规，不代表审批、编译或执行成功。

【步骤 3】检查风险字段由谁决定
Agent 的工具输入是否包含 irreversible： False
说明：False 表示 Agent 不能自行声明风险；它不表示服务端已判定本次操作可逆。

【步骤 4】把中文展示词“加列”误填为机器动作
页面展示词：加列；机器协议值：add_column。
第 1 步，字段 action：action 必须是 ('create_table', 'add_column', 'alter_column', 'create_index', 'drop_column', 'drop_table') 之一
说明：系统拒绝的是不符合机器协议的

&emsp;&emsp;运行输出需要分阶段解释：

1. **管理员原话**：证明探针明确展示了业务输入，没有把输入藏进 `title` 或 `summary`。
2. **完整 JSON**：证明申请可以序列化为可检查形状；它不是 SQL，也不表示申请已经提交或数据库已经修改。
3. **字段契约问题数量为 `0`**：只表示当前申请通过 `step_issues()` 的字段契约检查，可以进入后续审查；不表示审批、编译、预演或执行成功。
4. **`irreversible` 为 `False`**：只表示 Agent 工具输入中没有这个字段，不表示服务端已经把操作判定为可逆。
5. **“加列”被定位为 `action` 错误**：系统拒绝的是不符合机器协议的字段值，不是拒绝中文页面；界面可以展示“加列”，内部必须提交 `add_column`。
6. **`skill_id='add-column'`**：这里只是申请中的依据标识。探针没有调用完整提交函数，不能证明本轮真实读取过该 Skill。

&emsp;&emsp;这段探针可以证明管理员需求能被表示为字段明确的结构化申请，`add_column` 在当前 `table_structure` v2 契约中通过字段检查，非法动作能定位到第 1 步的 `action` 字段，且 Agent 不能自行携带 `irreversible`。它不能证明真实模型已经理解需求、Skill 已真实读取、申请已提交或审批、SQL 已生成、预演或执行，也不能证明 `ecommerce_core.orders.service_level` 已真实创建。

&emsp;&emsp;配套 Notebook 还提供“会员等级”真实模型探针。模型读取变更总纲、Skill 清单和对象事实后，最终选择 `submit_clarification`，追问会员等级的权威数据来源、目标对象和允许取值。这个结果展示了“信息不足时先澄清”的一次真实工具调用分支；它没有生成完整 `ChangeIntent`，没有执行 SQL，也不能证明后续数据库变更成功。

### 2.3 学习检查

1. 管理员说“给订单表增加服务等级字段”后，模型允许交出哪三类结果？为什么不能直接执行 SQL？
2. `title`、`summary` 与 `values` 分别服务谁？为什么给人看的说明不能替代机器字段？
3. `read_skill('add-column')` 与 `skill_id='add-column'` 有什么区别？为什么只看到 `skill_id` 不能证明 Skill 已真实读取？
4. 为什么 Agent 工具输入没有 `irreversible`？字段契约问题数量为 `0` 又为什么不等于变更已经成功？

<details><summary>参考要点</summary>

1. 信息不足时澄清，确认无需变化时提交无需变更，信息足够时提交 `ChangeIntent`。模型只整理申请，没有数据库执行工具；后续技术检查和执行必须由确定性链路完成。
2. `title` 与 `summary` 帮助人理解整份申请和单个步骤；`values` 保存动作、目标对象和具体参数，供确定性代码检查。人类说明不能替代稳定机器协议。
3. `read_skill` 是实际读取操作，成功后会留下本轮读取记录；`skill_id` 是申请中的依据标识。只有完整提交路径结合 `skills_read` 才能核对“声明的 Skill 是否真的读过”。
4. 风险由服务端根据动作规则判断，不能由申请方自报。问题数量为 `0` 只说明当前字段契约合规，不证明模型理解、审批、编译、预演、数据库权限或执行结果。

</details>

&emsp;&emsp;一句话收束：**管理员提供业务目标，模型把它整理成有依据、可检查的申请；确定性代码独立检查字段和风险，申请本身没有数据库执行能力。**

## <center>第三章：管理员变更如何经过检查、预演和确认后安全执行</center>

&emsp;&emsp;第二章结束时，系统拿到的是一份结构清楚的变更申请，数据库还没有发生变化。第三章接住这份申请，解决一个更关键的问题：**系统怎样确认申请是合法的、计划是真的能执行的，而且管理员最后执行的仍然是刚才检查和预演过的同一份内容。**

&emsp;&emsp;本章沿着“受理 → 意图识别 → 审校 → 编译 → 预演 → 放行 → 执行”七道工序前进。学习重点不是背诵七个名词，而是看清三类职责：模型负责把需求整理成申请；确定性代码负责审校、编译、冻结和重验；管理员保留最终确认权。完成本章后，你应能说明一次安全变更为什么不能从 Agent 直接跳到 SQL 执行，也能解释修改申请后为什么必须重新预演。

> **一句话理解第三章**：先把申请检查并翻译成确定的执行计划，再用真实事务预演；只有预演通过、计划没有变化且管理员当次确认后，系统才允许正式执行。

### 3.1 一份结构化申请为什么仍然不能直接执行？

&emsp;&emsp;先把本章会反复出现的词翻译成白话：

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 3-1　第三章关键术语的白话解释</font></p>

<div class="courseware-table-wrap">

| 源码或界面名称 | 白话解释 | 它回答的问题 |
| --- | --- | --- |
| 审校（review） | 检查申请本身是否完整、条件是否成立、步骤顺序是否合理 | “这份申请能不能成立？” |
| 编译（compile） | 用确定性代码把申请转换成 SQL、领域包补丁等执行计划 | “具体要怎样改？” |
| 自检（self-check） | 对编译结果再做一次机器规则检查 | “代码生成的计划有没有越界或漏项？” |
| 预演（dry-run） | 在真实 PostgreSQL 事务中按顺序执行，但最后强制回滚 | “这份计划在真实数据库条件下跑不跑得通？” |
| 变更冻结（freeze） | 把已经检查好的计划及其依赖版本封成一个不可偷换的版本 | “后面用的还是不是刚才那一份？” |
| 指纹（fingerprint） | 根据冻结内容计算出的摘要；绑定内容改变，指纹也会改变 | “两份冻结内容是否完全一致？” |
| 有效期（TTL） | 冻结件允许继续使用的时间，本项目为 15 分钟 | “这份检查结果是否已经过期？” |
| 放行（Gate） | 执行前集中检查所有门槛 | “现在是否满足正式执行条件？” |
| 拒绝优先（fail-closed） | 不能证明安全时默认不执行 | “信息不一致时应该冒险继续，还是停下来？” |

</div>

&emsp;&emsp;七道工序各自处理的问题并不相同：

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 3-2　七道受控工序各自解决什么问题</font></p>

<div class="courseware-table-wrap">

| 工序 | 主要执行者 | 输入与输出 | 失败时发生什么 |
| --- | --- | --- | --- |
| 1. 受理 | 确定性服务代码 | 接收管理员需求，建立变更单 | 请求不能进入后续流程 |
| 2. 意图识别 | Agent + 受限工具 | 把自然语言整理成结构化变更申请 | 继续澄清或记录意图失败 |
| 3. 审校 | 确定性审校代码 | 检查结构、前置条件和步骤顺序 | 返回具体问题，不进入编译 |
| 4. 编译 | 各变更族的确定性处理器 | 生成 SQL、领域包补丁、影响范围和冻结件 | 编译或自检失败，不生成可执行计划 |
| 5. 预演 | PostgreSQL 执行器 | 真实执行整份计划并收集反馈，最后回滚 | 保存失败步骤和诊断，不进入放行 |
| 6. 放行 | 门槛代码 + 管理员确认 | 检查冻结、预演和当次确认 | 任一条件不满足就拒绝执行 |
| 7. 执行 | PostgreSQL 执行器 | 正式提交同一冻结计划，回读终态并写审计 | 进入执行失败终态并保留记录 |

</div>

&emsp;&emsp;先用一张图把七道工序和失败出口连起来。冻结是编译产生的受控计划，人工确认位于预演之后、正式放行之前；二者都不能被画成独立于主链的装饰步骤。

<!-- IMAGE_PROMPT
Use case: scientific-educational
Asset type: 16:9 横向课件七阶段可信写链流程图
Primary request: 教学目的为按顺序理解可信写链和失败出口。阅读顺序严格从左到右：受理→意图识别→审校→编译→5 预演→人工确认→6 放行→7 执行。冻结计划必须作为“编译产物”附着在编译节点下方，并以约束线连接预演和执行前；人工确认是进入 Gate/放行的输入，不得画在放行之后；执行后连接终态验证与审计。每个主节点可有灰红失败出口。
Text (verbatim): "受理"; "意图识别"; "审校"; "编译"; "5 预演"; "人工确认"; "6 放行"; "7 执行"; "冻结计划（编译产物）"; "事务执行后回滚"; "终态验证"; "审计"; "结构/前置条件/顺序失败"; "Policy 或自检失败"
Color palette: 主链蓝绿色；冻结计划为蓝色附着件；人工确认橙色；失败出口灰红；终态验证绿色。
Constraints: 七个主节点只能是上述七项，绝对不得增加第八个“冻结”；不得把 dry-run 画成静态 SQL 预览；人工确认必须位于预演（5）与放行（6）之间，作为进入 Gate/放行的输入，不得放在放行之后、悬空或被旁路；不得省略预演或人类确认。中文清晰无错别字，技术标识符保持英文原文。
Avoid: 额外英文装饰、logo、水印、3D、卡通风。
Style/medium: scientific-educational，浅色背景、清楚箭头。
-->

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164325582.png" alt="管理员可信写链七道工序" width="85%"></div>

&emsp;&emsp;审校分三趟进行。第一趟检查结构，例如步骤字段是否完整；结构不成立时不会继续猜测它的业务含义。第二趟检查前置条件，例如目标表和列是否符合当前真实结构。第三趟检查顺序，例如依赖步骤是否排在提供条件的步骤之后。申请通过后，编译器才按照变更类别生成技术计划；编译后的自检继续检查语句类型、允许操作的 schema、参数绑定、业务数据条件和不可逆标记。

&emsp;&emsp;**预演不是 SQL 预览。**源码中的 PostgreSQL 执行器会开启事务、切换受控角色、锁定受影响对象、重验冻结前提，再按依赖顺序真实运行每一步。每个步骤使用嵌套事务保存独立反馈，最外层事务使用 `force_rollback=True`，无论步骤成功还是失败，最后都不提交。回滚后还会再次读取关键状态；如果发现仍有数据或结构被持久化，系统会把它视为严重失败。

&emsp;&emsp;**变更冻结也不是第八道工序。**它是第四步编译产生的“密封执行计划”。可以把它理解成一份已经盖章的施工方案：里面绑定了变更意图、执行计划、领域版本、数据修订版本、数据库结构版本、语义版本、Policy 版本、部署安全策略和责任主体。预演和正式执行都要使用同一个冻结件，而不是重新从草稿临时生成计划。

&emsp;&emsp;冻结件的有效期是 15 分钟。执行前，如果冻结过期，或者绑定的目标对象、版本与当前事实不一致，系统无法证明“现在执行的仍是刚才确认的内容”，因此会直接拒绝继续，这就是“拒绝优先”。这里只检查与本次计划绑定的前提；一个完全无关的对象发生变化，不代表当前冻结一定失效。

> **记住冻结有效期**：冻结就是把审校、编译完成的执行计划连同当时的数据结构、领域版本、Policy 和责任主体一起密封。15 分钟内，系统允许拿这份密封计划做预演和执行；超过 15 分钟，计划记录还在，但它作为执行凭证的资格失效，旧 dry-run 也不能继续放行，必须重新生成冻结件并重新预演。

#### 修改申请后，旧预演为什么必须作废？

&emsp;&emsp;管理员修改步骤后，系统会重新走审校和编译。这里必须区分两种结果：

1. **编辑被拒绝**：新值没有保存，原申请和原冻结内容保持不变；系统不能偷偷使用被拒绝的新值。
2. **编辑成功保存**：新计划会产生新指纹，旧冻结立即失效，旧冻结对应的预演结果也不能继续用于放行；管理员必须针对新冻结重新预演。

&emsp;&emsp;放行阶段会检查预演是否通过，并把“管理员发起本次执行请求”作为这一次的确认。若计划包含不可逆对象，还必须准确输入该对象名称完成二次确认。它不是一个可以永久保存的 `admin_confirmed=true` 开关。执行接口重复收到同一冻结件时，会优先返回已有执行收据，避免把一次成功变更当作两次不同操作再次执行。

&emsp;&emsp;回到服务等级案例：第二章提交“给订单增加服务等级”的申请后，第三章先检查表、字段、类型和步骤依赖，再编译出确定性计划并预演。如果管理员把列名从 `service_level` 改成 `service_tier` 且编辑成功，旧冻结与旧预演就必须作废；不能拿旧列名的预演结果去放行新列名的正式执行。

<!-- IMAGE_PROMPT
Use case: infographic-diagram
Asset type: 16:9 横向课件状态迁移图
Primary request: 教学目的为解释结构化字段编辑为何会使旧变更冻结和对应 dry-run 失效，并显示其他拒绝入口。阅读顺序：有效冻结+已完成 dry-run→提交结构化字段编辑→重新审校/编译，分成两条明确分支：编辑被拒绝→新值不生效→原内容保持；编辑成功保存→旧变更冻结与 dry-run 失效→新冻结→新 dry-run→执行请求。另从侧方连入两条失效/拒绝入口：TTL 到期；执行前重验发现绑定前提漂移。
Text (verbatim): "有效冻结"; "已完成 dry-run"; "提交结构化字段编辑"; "重新审校/编译"; "编辑被拒绝"; "新值不生效"; "原内容保持"; "编辑成功保存"; "旧变更冻结与 dry-run 失效"; "TTL 到期"; "执行前重验发现绑定前提漂移"; "新冻结"; "新 dry-run"; "执行请求"
Color palette: 有效状态蓝绿色；人工门槛橙色；被拒绝分支灰红；失效状态灰色虚线；新链恢复为蓝绿色。
Constraints: 必须包含两条主分支和两个侧向失效/拒绝入口及精确标签；侧向入口指向失效或拒绝执行，不得暗示无关对象变化必然失效；不得画永久确认；不得出现任何额外 API 字段标签。中文清晰无错别字，技术标识符保持英文原文。
Avoid: 额外英文装饰、logo、水印、3D、卡通风。
Style/medium: infographic-diagram，浅色背景、状态箭头明确。
-->

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164215416.png" alt="结构化字段编辑后的冻结失效与重新放行" width="85%"></div>

#### 服务等级案例中，管理员究竟要看几个步骤

&emsp;&emsp;这里需要区分三种不同的“步骤数”。七道工序是整张变更单从受理到执行的生命周期；三趟审校是系统代码对申请进行的结构、前置条件和顺序检查；而服务等级主案例本身包含 **6 个业务变更步骤**：

1. 给 `ecommerce_core.orders` 增加 `service_level` 字段；
2. 按 `order_amount` 把历史订单确定性回填为 `L1`、`L2`；
3. 给 `analytics.v_paid_order_items` 增加服务等级输出列；
4. 给 `analytics.v_refund_items` 增加服务等级输出列；
5. 在领域包中创建 `service_level` 维度；
6. 在领域包中创建 `service_sales` 指标，并声明它支持服务等级维度。

&emsp;&emsp;管理员在页面上检查的是这张六步申请的目标、顺序、依赖、影响面、不可逆标记和 dry-run 结果，最后对冻结后的**整份计划**发起一次执行确认，并不是逐个步骤点击六次“批准”。系统审校器则会对整份意图完成三趟检查，并在编译阶段对每一个步骤分别编译和自检。

#### 编译生成的 SQL 最后怎样执行

&emsp;&emsp;编译阶段会依次遍历六个步骤，根据类别选择对应处理器。表结构、业务数据和治理视图步骤会生成 `CompiledStep(kind="sql")` 或复合步骤；语义维度、指标步骤生成领域包补丁，不会伪装成 SQL。每个 SQL 步骤可以包含一条或多条带参数的 `SqlStatement`，随后与依赖关系、影响对象、预期终态一起写入冻结计划。

&emsp;&emsp;dry-run 时，执行器从冻结件恢复这份计划，在强制回滚的外层事务中按 `step_order` 执行。每条 SQL 最终通过 `connection.execute(statement.statement, dict(statement.parameters))` 运行；DDL 会先切换到受控对象属主角色。某一步失败后，依赖它的后续步骤会标记为跳过，与它无依赖的步骤仍可继续收集预演反馈，最后整笔事务回滚。

&emsp;&emsp;正式执行时，Gate 先确认冻结和 dry-run 仍有效，执行器再开启一个“整单原子事务”，重新检查冻结前提并按顺序执行计划。正式执行不为每一步单独吞掉错误：任意一步失败都会使整单事务回滚；全部成功后，数据修订、领域版本和执行收据才随事务一起提交。这样保证六步要么整体成功，要么不留下半套服务等级能力。

&emsp;&emsp;源码锚点：[审校服务](HermesAnalytics-第三版/backend/src/hermes_analytics/admin_change/review/service.py)、[编译自检](HermesAnalytics-第三版/backend/src/hermes_analytics/admin_change/compile/selfcheck.py)、[冻结机制](HermesAnalytics-第三版/backend/src/hermes_analytics/admin_change/freeze.py)、[服务入口](HermesAnalytics-第三版/backend/src/hermes_analytics/application/admin_changes/service.py)。

### 3.2 执行前的 Gate 到底检查哪些条件？

&emsp;&emsp;上一节已经得到审校、编译后的冻结计划。本段只观察执行前 Gate 的四项纯规则输入：dry-run 状态、当次人工确认、计划是否包含不可逆动作，以及不可逆对象名是否已经准确输入。它不会创建真实冻结件、运行 dry-run、模拟 15 分钟经过，也不会连接数据库。

&emsp;&emsp;下面的探针把四项 Gate 输入组合成五个正反场景。运行时不要只看最终的 `passed`，而要逐项比较 dry-run 状态、当次人工确认、是否包含不可逆动作，以及对象名二次确认怎样共同决定放行或拒绝。

In [4]:
"""展示放行门槛怎样组合判断，并打印每个拒绝原因。"""
from hermes_analytics.domain.admin_change import (
    ChangeDryRunState,
    ConfirmationGate,
    describe_unmet_reasons,
)
from hermes_analytics.admin_change.freeze import FREEZE_TTL

cases = {
    '尚未预演（即使已发起确认）': ConfirmationGate(
        dry_run_state=ChangeDryRunState.NOT_RUN,          # 当前冻结计划还没有通过 dry-run。
        admin_confirmed=True,                             # 模拟管理员已发起本次执行请求。
        irreversible=False,                               # 当前计划不含不可逆动作。
        irreversible_object_name_entered=False,           # 普通变更不需要输入对象名。
    ),

    '预演通过但管理员尚未确认': ConfirmationGate(
        dry_run_state=ChangeDryRunState.PASSED,           # 当前冻结计划已经通过 dry-run。
        admin_confirmed=False,                            # 尚未形成这一次的人工确认。
        irreversible=False,                               # 当前计划不含不可逆动作。
        irreversible_object_name_entered=False,           # 普通变更不需要输入对象名。
    ),

    '普通变更已预演并确认': ConfirmationGate(
        dry_run_state=ChangeDryRunState.PASSED,           # 当前冻结计划已经通过 dry-run。
        admin_confirmed=True,                             # 管理员已发起本次执行请求。
        irreversible=False,                               # 当前计划不含不可逆动作。
        irreversible_object_name_entered=False,           # 普通变更不需要输入对象名。
    ),

    '不可逆变更未输入对象名': ConfirmationGate(
        dry_run_state=ChangeDryRunState.PASSED,           # 当前冻结计划已经通过 dry-run。
        admin_confirmed=True,                             # 管理员已发起本次执行请求。
        irreversible=True,                                # 当前计划包含不可逆动作。
        irreversible_object_name_entered=False,           # 尚未输入准确对象名完成二次确认。
    ),

    '不可逆变更完成二次确认': ConfirmationGate(
        dry_run_state=ChangeDryRunState.PASSED,           # 当前冻结计划已经通过 dry-run。
        admin_confirmed=True,                             # 管理员已发起本次执行请求。
        irreversible=True,                                # 当前计划包含不可逆动作。
        irreversible_object_name_entered=True,            # 已输入准确对象名完成二次确认。
    ),
}

freeze_minutes = int(FREEZE_TTL.total_seconds() // 60)
print('【上下文】已经审校和编译的计划进入执行前 Gate')
print(f'冻结件有效期配置：{freeze_minutes} 分钟')
print('说明：这里只读取 TTL 配置值，没有模拟时钟流逝或验证某个冻结件是否过期。')

print('\n【Gate 逐项判断】')
for index, (name, gate) in enumerate(cases.items(), start=1):
    message = '允许执行' if gate.passed else describe_unmet_reasons(gate.unmet_reasons)
    reasons = list(gate.unmet_reasons) if gate.unmet_reasons else ['无']
    print(f'\n场景 {index}：{name}')
    print(
        '输入状态：'
        f'dry_run={gate.dry_run_state.value}, '
        f'admin_confirmed={gate.admin_confirmed}, '
        f'irreversible={gate.irreversible}, '
        f'object_name_entered={gate.irreversible_object_name_entered}'
    )
    print('未满足原因码：', reasons)
    print(f'放行结果：passed={gate.passed} -> {message}')

print('\n【规则总结】')
print('普通变更：dry-run 通过 + 管理员当次确认，才允许执行。')
print('不可逆变更：还必须输入准确对象名，完成额外二次确认。')
print('【本段边界】真实服务还会验证活动冻结件、指纹、TTL、绑定版本和受影响对象。')

【上下文】已经审校和编译的计划进入执行前 Gate
冻结件有效期配置：15 分钟
说明：这里只读取 TTL 配置值，没有模拟时钟流逝或验证某个冻结件是否过期。

【Gate 逐项判断】

场景 1：尚未预演（即使已发起确认）
输入状态：dry_run=not_run, admin_confirmed=True, irreversible=False, object_name_entered=False
未满足原因码： ['dry_run_not_passed']
放行结果：passed=False -> 预演尚未通过，请先完成预演并确认结果正常后再执行

场景 2：预演通过但管理员尚未确认
输入状态：dry_run=passed, admin_confirmed=False, irreversible=False, object_name_entered=False
未满足原因码： ['confirmation_missing']
放行结果：passed=False -> 本次执行尚未确认，请重新发起一次执行请求完成确认

场景 3：普通变更已预演并确认
输入状态：dry_run=passed, admin_confirmed=True, irreversible=False, object_name_entered=False
未满足原因码： ['无']
放行结果：passed=True -> 允许执行

场景 4：不可逆变更未输入对象名
输入状态：dry_run=passed, admin_confirmed=True, irreversible=True, object_name_entered=False
未满足原因码： ['irreversible_object_name_missing']
放行结果：passed=False -> 这是不可逆操作，请先在确认框内输入对象名称完成二次确认

场景 5：不可逆变更完成二次确认
输入状态：dry_run=passed, admin_confirmed=True, irreversible=True, object_name_entered=True
未满足原因码： ['无']
放行结果：passed=True -> 允许执行

【规则总结】
普通变更：dry-run 通过 + 管理员当次确认，才允许执行。
不可

&emsp;&emsp;运行输出应按下面五种情况理解：

1. 尚未预演时，即使管理员已经发起确认，也会得到 `dry_run_not_passed`。
2. dry-run 已通过但没有当次确认时，会得到 `confirmation_missing`。
3. 普通变更只有在 dry-run 通过且管理员当次确认后，`passed` 才为 `True`。
4. 不可逆计划还必须准确输入对象名，否则得到 `irreversible_object_name_missing`。
5. `FREEZE_TTL=15 分钟` 只是源码配置值；本探针没有验证某个真实冻结件是否已经过期。

&emsp;&emsp;`passed=True` 只表示这四项纯规则条件满足，不表示活动冻结件、指纹、绑定版本和受影响对象已经完成真实服务重验，更不表示数据库已经执行成功。

### 3.3 放行后的参数化 SQL 怎样进入受限执行器？

&emsp;&emsp;下面的输入已经是确定性编译器生成的 `CompiledStep`，不是 Agent 提交的自由 SQL。记录连接只观察执行器内部调用顺序，不访问 PostgreSQL；固定的 `rowcount=2` 只是模拟值。

In [5]:
"""记录真实执行器方法的调用顺序，不连接 PostgreSQL。"""
from hermes_analytics.admin_change.models import CompiledStep, SqlStatement
from hermes_analytics.infrastructure.postgres.admin_executor import PostgresAdminExecutor


class RecordingCursor:
    rowcount = 2  # 模拟数据库游标报告影响 2 行，不是真实数据库结果。


class RecordingConnection:
    def __init__(self):
        self.calls = []  # 按发生顺序保存 statement 与 parameters。

    def execute(self, statement, parameters=None):
        self.calls.append((statement, parameters))  # 只记录调用，不发送到 PostgreSQL。
        return RecordingCursor()                    # 返回固定 rowcount 的模拟游标。


connection = RecordingConnection()                    # 用记录对象替代真实 psycopg 连接。
executor = object.__new__(PostgresAdminExecutor)      # 跳过完整初始化，只调用目标内部方法。
step = CompiledStep(
    step_order=2,                                     # 这是冻结计划中的第 2 个编译步骤。
    kind='sql',                                       # 表示当前步骤走物理 SQL 执行分支。
    statements=(                                      # 一个步骤可以包含一条或多条参数化语句。
        SqlStatement(
            statement=(                               # SQL 模板保留参数占位符，不拼接业务值。
                'UPDATE ecommerce_core.orders '
                'SET service_level = %(level)s '
                'WHERE service_level IS NULL'
            ),
            parameters={'level': 'L1'},               # 业务值通过独立参数字典绑定。
        ),
    ),
)

statement_input = step.statements[0]
print('【输入】确定性编译器生成的 SQL 步骤')
print(f'步骤序号：{step.step_order}；步骤类型：{step.kind}')
print('SQL 模板：', statement_input.statement)
print('绑定参数：', dict(statement_input.parameters))
print('说明：SQL 模板与业务值分开保存，当前输入不是 Agent 提交的自由 SQL。')

print('\n【步骤 1】调用真实执行器的物理步骤方法')
rows, message = executor._execute_physical_step(connection, step)
print('模拟累计影响行数：', rows)
print('测量消息：', message)
print('说明：None 表示本步骤没有额外 measurement 消息，不表示执行失败。')

print('\n【步骤 2】记录连接收到的调用顺序')
for index, (statement, parameters) in enumerate(connection.calls, start=1):
    print(f'{index}. statement={statement}')
    print(f'   parameters={parameters}')

print('说明：第 1 次切换受控角色，第 2 次执行参数化 SQL，第 3 次恢复角色。')
print('【本段边界】记录连接没有开启事务、访问 PostgreSQL、提交或回滚；影响 2 行是模拟值。')

【输入】确定性编译器生成的 SQL 步骤
步骤序号：2；步骤类型：sql
SQL 模板： UPDATE ecommerce_core.orders SET service_level = %(level)s WHERE service_level IS NULL
绑定参数： {'level': 'L1'}
说明：SQL 模板与业务值分开保存，当前输入不是 Agent 提交的自由 SQL。

【步骤 1】调用真实执行器的物理步骤方法
模拟累计影响行数： 2
测量消息： None
说明：None 表示本步骤没有额外 measurement 消息，不表示执行失败。

【步骤 2】记录连接收到的调用顺序
1. statement=SET ROLE hermes_object_owner
   parameters=None
2. statement=UPDATE ecommerce_core.orders SET service_level = %(level)s WHERE service_level IS NULL
   parameters={'level': 'L1'}
3. statement=RESET ROLE
   parameters=None
说明：第 1 次切换受控角色，第 2 次执行参数化 SQL，第 3 次恢复角色。
【本段边界】记录连接没有开启事务、访问 PostgreSQL、提交或回滚；影响 2 行是模拟值。


&emsp;&emsp;运行输出显示三次调用依次是 `SET ROLE hermes_object_owner`、参数化 `UPDATE`、`RESET ROLE`。这证明当前内部方法把 SQL 模板和参数字典分开传给连接对象，并在成功分支恢复角色；不能证明真实 PostgreSQL 权限、事务提交或回滚、冻结重验、终态回读，也不能证明订单表真实更新了两行。

### 3.4 现有测试记录能证明到哪一层？

&emsp;&emsp;运行目录是 `HermesAnalytics-第三版/backend`，环境由项目 `uv run --frozen` 管理。先运行组件契约，检查数据模型、三趟审校、编译自检、冻结生命周期与状态机：

```bash
PYTHONDONTWRITEBYTECODE=1 uv run --frozen python -m pytest -p no:cacheprovider -q \
  tests/unit/admin_change/test_models.py \
  tests/unit/admin_change/test_review_service.py \
  tests/unit/admin_change/test_compile_selfcheck.py \
  tests/unit/admin_change/test_freeze_lifecycle.py \
  tests/unit/test_admin_change_state_machine.py
```

&emsp;&emsp;当前保存的组件结果为 `175 passed`。隔离 PostgreSQL 17 跨层测试文件的最新结果为 `26 passed`，覆盖冻结前提与对象指纹、数据库隐式副作用的失败关闭、正向六步与反向四步各提交一次、controlled delete、三处能力验真，以及六个正向失败点的整单回滚。两层结果必须分开理解：

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 3-3　管理员链测试结果与证明边界</font></p>

<div class="courseware-table-wrap">

| 测试层级 | 当前结果 | 可以证明 | 仍不能证明 |
| --- | --- | --- | --- |
| 组件契约 | `175 passed` | 数据模型、三趟审校、编译自检、冻结生命周期与状态机的定向契约通过 | PostgreSQL 权限、真实事务或部署栈已经通过 |
| 隔离 PostgreSQL 17 | `26 passed` | B18 T6 跨层测试文件中的冻结前提、正反向计划、隐式副作用拒绝、三处验真与失败回滚在隔离 fixture 中通过 | 共享演示库、任意变更类型、浏览器或真实模型端到端链路已经通过 |

</div>

&emsp;&emsp;看到 `PASS` 时，先问“这是哪一层、运行了哪些用例、使用了什么环境”。组件测试不能代替数据库测试；隔离数据库测试也不能代替 Docker/API、浏览器和真实模型验收。

### 3.5 学习检查

1. 冻结在七道工序中是什么位置？为什么不能单列为另一道？
2. 编辑被拒绝时，系统为什么不能让新值影响任何后续执行？
3. 为什么预演通过后，管理员修改一个字段仍然必须重新预演？
4. dry-run 与“只把 SQL 打印出来看看”有什么根本区别？

<details><summary>参考要点</summary>

1. 变更冻结是编译产物，并约束预演与执行前的管理员写计划；它不是独立工序。它会在 TTL 到期、成功编辑改变绑定指纹，或执行前重验发现绑定前提漂移时失效或拒绝执行。
2. 拒绝表示新结构化值未成功保存，继续使用它会让已审校、已预演和实际执行的内容错配。
3. 成功编辑会生成内容不同的新计划和新指纹，旧 dry-run 证明的是旧计划可以执行，不能证明新计划也可以执行。
4. SQL 预览只展示文本；本项目的 dry-run 会在真实 PostgreSQL 事务、真实权限和真实结构条件下执行步骤、收集影响与终态反馈，然后强制回滚。

</details>

### 3.6 完成第三章后，应该守住哪些边界？

1. **第三章处理的是安全执行，不再只是理解需求。**第二章的申请必须经过审校、编译、预演、放行和执行，才可能真正改变数据库。
2. **Agent 不负责生成并直接执行自由 SQL。**模型形成结构化申请后，SQL、领域包补丁、风险检查和执行顺序由确定性代码负责。
3. **审校、编译自检和 dry-run 解决不同问题。**审校检查申请，编译自检检查生成计划，dry-run 检查计划在真实数据库条件下能否运行。
4. **dry-run 是真实执行后回滚，不是静态预览。**它使用真实事务、权限、锁和数据库结构收集反馈，外层事务保证不提交。
5. **冻结件保证预演与执行针对同一份内容。**它绑定计划、版本、Policy、责任主体和受影响对象，并以指纹识别内容是否变化。
6. **修改成功就必须重新冻结、重新预演。**被拒编辑不保存；成功编辑改变指纹，使旧冻结及其 dry-run 失去放行资格。
7. **最终执行权仍由管理员掌握。**当次执行请求构成人工确认，不可逆操作还要输入准确对象名；任何门槛无法证明满足时，系统默认拒绝执行。
8. **测试证据必须按层级表达。**组件结果是 `175 passed`，隔离 PostgreSQL 17 跨层测试文件是 `26 passed`；它们不能替代共享数据库、Docker/API、浏览器或真实模型验收。

> **一句话收束**：Hermes 管理员链的核心不是“让 Agent 会改数据库”，而是让 Agent 只表达意图，再由可检查、可预演、可冻结、可确认和可审计的确定性链路完成真实变更。

&emsp;&emsp;受控计划执行并得到终态记录后，还要继续确认新字段、治理视图和领域版本是否共同进入分析师的新运行态。只有这一步通过，管理员变更才从“写入成功”升级为“新的分析能力可用”。

## <center>第四章：如何确认分析能力真正可用，并在证据不足时拒绝猜测</center>

&emsp;&emsp;第三章结束时，一份受控变更计划已经经过检查、预演、确认和执行。但“执行成功”只是写链终点，不是分析能力终点。第四章要建立一套更严格的判断方法：**已经落库的能力要逐层验真；仍然缺失的能力不能靠模型猜测；一次成功经验也不能未经人工裁决就变成下一次规则。**

&emsp;&emsp;本章用两个相反的需求完成这三次判断：服务等级是正向案例，用来确认一项能力怎样真正进入分析师运行态；会员等级是反向案例，用来观察系统发现业务事实不足时怎样停下来澄清。最后再检查 `PENDING` 提案，确认“做成功一次”不等于“以后自动照做”。

> **一句话理解第四章**：表里有、视图露、语义认、新分析会用，才能宣布能力生效；缺少来源或规则时，Agent 必须澄清；可复用经验仍要经过人工批准。

建议按下面的顺序学习：

1. 用服务等级案例看清能力生效的四个阶段；
2. 沿六步计划追踪业务表回填、治理视图重建和领域版本激活；
3. 在三个证据位置分别验真；
4. 用隔离 PostgreSQL 用例检查成功提交与失败回滚；
5. 用会员等级需求观察真实模型怎样识别能力缺口；
6. 沿五个源码入口理解三个 `submit_*` 工具怎样被确定性代码约束；
7. 用 `PENDING` 探针确认经验提案不能越级生效。

### 4.1 字段已经写进表，为什么分析师仍然不能直接使用？

&emsp;&emsp;一个字段要成为分析师可用的能力，必须连续经过四个阶段。任何一层缺失，系统都只能说“部分内容已经存在”，不能宣布能力已经可用。

1. **业务表有事实**：`ecommerce_core.orders` 增加 `service_level`，并按确定性规则回填 `L1`、`L2`。
2. **治理视图有入口**：`analytics.v_paid_order_items` 与 `analytics.v_refund_items` 都暴露同一字段，让销售与退款分析从受控入口读取它。
3. **活动语义能理解**：激活领域版本登记 `service_level` 维度、`service_sales` 指标及其关系。
4. **新分析实际会用**：分析师的新规划输入加载活动版本，并按服务等级返回 `L1`、`L2` 两组结果。

> **标识说明**：`service_level` 和 `service_sales` 是服务等级案例的正式业务标识；测试文件名中的 `test_b18_t6...` 仍是内部测试批次标识，不属于字段、维度或指标名称。

<div align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164215503.png" alt="一项字段经过业务表、治理视图、激活领域版本和新分析运行态后成为分析能力" width="85%"></div>

&emsp;&emsp;这张图只回答一个问题：一项字段怎样变成分析能力。主流程从业务表、治理视图、语义层走到新分析；底部三个徽章表示需要核对的证据位置；左下角的旧分析 Query 是旁路，说明已经冻结的旧查询结果不会被新字段反向改写。

### 4.2 Agent 提交以后，六步计划怎样真正落到三层？

&emsp;&emsp;先纠正一个容易混淆的顺序：Agent 提交的是结构化 `ChangeIntent`，不是已经执行的数据库动作；管理员最终确认的也不是 Agent 的一段自然语言，而是**已经完成审校、编译、自检、dry-run 和冻结的同一份计划**。因此，从“Agent 已经提交”到“分析师能够查询”，中间还要经过下面这条完整链路：

```text
Agent 提交六步 ChangeIntent
→ 确定性代码审校并按步骤类型选择编译器
→ 生成 4 个 SQL 步骤与 2 个领域包补丁
→ 汇总候选 manifest（语义清单）、数据修订候选和预期终态
→ 冻结并完成 dry-run
→ 管理员确认执行
→ 同一数据库事务内执行六步、发布修订、激活领域版本并核对终态
→ 新分析会话读取新的活动版本并生成查询
```

&emsp;&emsp;其中，`AdminChangeService.receive_intent()` 把 Agent 的结构化申请交给 `_review_compile_and_freeze()`。这段确定性代码会按 `TABLE_STRUCTURE`、`BUSINESS_DATA`、`GOVERNANCE_VIEW`、`SEMANTICS` 四类步骤选择对应处理器；每一步先编译、自检，再把结果投影到下一步要看到的结构快照和领域目录。这样，第 5 步创建维度时已经能看到第 3 步投影出的新视图字段，第 6 步创建指标时也已经能看到新维度。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 4-1　服务等级六步计划分别生成什么</font></p>

<div class="courseware-table-wrap">

| 顺序 | 结构化动作 | 编译产物 | 真正生效的位置 |
| --- | --- | --- | --- |
| 1 | 给 `ecommerce_core.orders` 增加 `service_level` | `ALTER TABLE ... ADD COLUMN ... text` | 业务表出现可空新列 |
| 2 | 按 `order_amount` 回填服务等级 | 参数化的 `WITH ... UPDATE` | 仅更新 `service_level IS NULL` 的订单：金额 `< 1000` 写 `L1`，其余写 `L2` |
| 3 | 给 `analytics.v_paid_order_items` 追加输出列 | 按服务端保存的 canonical（规范）定义生成完整 `CREATE OR REPLACE VIEW` | 销售分析治理入口暴露 `o.service_level` |
| 4 | 给 `analytics.v_refund_items` 追加输出列 | 同样生成完整 `CREATE OR REPLACE VIEW` | 退款分析治理入口暴露同一字段 |
| 5 | 创建 `service_level` 维度 | `DomainPackPatch(op="add", path="/dimensions/-", ...)` | 先进入整单候选 manifest，不在这一步单独发布 |
| 6 | 创建 `service_sales` 指标 | `DomainPackPatch(op="add", path="/metrics/-", ...)` | 候选 manifest 增加按 `time`、`service_level` 分组的销售额指标 |

</div>

&emsp;&emsp;第 2 步不是让模型拼接一条自由 SQL。`BusinessDataProcessor` 把类型化的 bucket 表达式编译成参数化 `CASE`：读取 `order_amount`，小于 `1000` 产生 `L1`，否则产生 `L2`；再通过主键 `order_id` 回连目标行，而且只回填当前字段仍为 `NULL` 的记录。数值阈值和等级值来自已经通过审校的结构化申请，作为绑定参数进入 SQL。

&emsp;&emsp;第 3、4 步也不是在原视图文本末尾随意追加字段。`GovernanceViewProcessor` 从结构快照读取服务端 canonical 定义，保留已有选择列、来源、过滤条件和视图选项，再加入 `o.service_level AS service_level`，最后生成完整的 `CREATE OR REPLACE VIEW`。因此，两份治理视图是被受控重建，而不是绕开既有定义临时创建另一条查询。

&emsp;&emsp;第 5、6 步与前四步有本质区别：`SemanticsProcessor` 产出的是领域包补丁，不是 SQL。编译阶段通过 `accumulate_candidate_manifest()` 按顺序把两个补丁应用到当前活动 manifest 的副本，形成一份完整候选 manifest。维度补丁登记字段来源、层级和筛选算子；指标补丁登记 `paid_amount` 求和公式、来源治理视图以及支持的分组维度。此时只是把“将要发布的新语义版本”冻结下来，活动版本还没有切换。

&emsp;&emsp;管理员确认后，`PostgresAdminExecutor` 打开一个外层事务，锁定受影响对象并复核冻结前提，然后依序处理六个编译步骤。第 1～4 步执行 DDL/DML；第 5～6 步只确认领域补丁已随计划携带，不在循环中各发布一次。六步都成功以后，执行器才发布新的数据修订，再调用 `publish_manifest()` 校验治理对象、写入 `system.domain_versions`，并把 `system.project_metadata.active_domain_version_id` 指向新版本。最后还要核对对象结构、数据终态、领域包摘要和版本增量；任一步、发布或终态核对失败，外层事务都会整单回滚。

> **关键边界**：管理员点“确认”不等于前三层已经成功；它只是授权执行冻结计划。只有执行事务提交、活动版本指针切换并通过终态核对，才说明新能力已经进入系统活动状态。

&emsp;&emsp;新分析平台也不会凭空“刷新出一个字段”。分析师提交新的分析问题后，新建的规划 operation 通过 `PostgresAnalysisPlanningRepository.load_planning_input()` 同时读取当前 `active_domain_version_id`、`active_data_revision_id` 和完整 manifest。领域包加载器由此识别 `service_sales` 指标及 `service_level` 维度，新的 FrozenQuery 才能以该指标、该分组和新版本坐标生成 SQL，并从治理视图读取结果。测试最终看到两组结果 `L1`、`L2`；已经冻结的旧 Query 仍绑定旧领域版本与旧数据修订，不会被这次发布反向改写。

&emsp;&emsp;沿源码阅读这条主线时，可以依次查看：[六步结构化申请与跨层测试](HermesAnalytics-第三版/backend/tests/integration/test_b18_t6_cross_layer_pg17.py)、[审校、编译与候选 manifest 汇总](HermesAnalytics-第三版/backend/src/hermes_analytics/application/admin_changes/service.py)、[表结构编译器](HermesAnalytics-第三版/backend/src/hermes_analytics/admin_change/compile/table_structure.py)、[业务数据编译器](HermesAnalytics-第三版/backend/src/hermes_analytics/admin_change/compile/business_data.py)、[治理视图编译器](HermesAnalytics-第三版/backend/src/hermes_analytics/admin_change/compile/governance_view.py)、[语义编译器](HermesAnalytics-第三版/backend/src/hermes_analytics/admin_change/compile/semantics.py)、[事务执行器](HermesAnalytics-第三版/backend/src/hermes_analytics/infrastructure/postgres/admin_executor.py)、[领域版本发布](HermesAnalytics-第三版/database/hermes_analytics_database/publish.py)和[新分析规划输入](HermesAnalytics-第三版/backend/src/hermes_analytics/infrastructure/postgres/analysis_planning_repository.py)。

> **一句话收束**：六步计划先被编译成“物理 SQL + 候选语义版本”，再在同一事务里完成业务表、治理视图、数据修订和领域版本发布；新分析会话读取新活动坐标后，服务等级才真正成为可查询能力。

### 4.3 四个阶段为什么只需要三个验真位置？

&emsp;&emsp;业务表和治理视图都是 PostgreSQL 中的物理对象，可以在同一个数据库位置核对；语义发布和分析使用则分别属于活动领域版本与分析师运行态，必须单独验证。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 4-2　新分析能力的三处验真</font></p>

<div class="courseware-table-wrap">

| 验真位置 | 必须看到的事实 | 可以证明 | 仍不能证明 |
| --- | --- | --- | --- |
| 物理数据库 | `orders` 新列与回填值存在；两份治理视图都输出该字段 | 业务事实与治理投影已经落库 | 活动语义已经发布，或分析规划已经采用它 |
| 激活领域版本 | `active_domain_version_id` 指向新版本；manifest 含新维度、指标及关联 | 当前语义入口已经切换到新版本 | 某次分析确实加载并使用了该版本 |
| 分析师运行态 | 新规划输入加载活动版本；结果出现 `L1`、`L2` 两组 | 该能力在一次受控分析中可以使用 | 浏览器、部署栈或真实模型完整链路已经验收 |

</div>

&emsp;&emsp;这里还有两种“冻结”不能混淆：旧分析 Query 的冻结保存既有查询和证据，不会被新变更反向改写；第三章的管理员变更冻结绑定待执行计划，计划内容、前提版本或有效期变化后必须重新冻结和预演。

### 4.4 怎样用隔离数据库测试验证正向能力和失败回滚？

&emsp;&emsp;下面的测试由 fixture 启动隔离 PostgreSQL 17，不连接共享演示库，也不是只检查 SQL 文本。把 `HERMES_TEST_PG17_BINDIR` 替换为本机 PostgreSQL 17 的二进制目录后，在 `HermesAnalytics-第三版/backend` 运行：

```bash
HERMES_TEST_PG17_BINDIR=/path/to/postgresql-17/bin python -m pytest -q -s \
  tests/integration/test_b18_t6_cross_layer_pg17.py
```

当前保存结果为：

```text
26 passed
```

&emsp;&emsp;这份测试文件还检查冻结前提、目标对象漂移、触发器和未建模外键副作用的失败关闭，并继续验证正向六步与反向四步各提交一次、controlled delete、三处能力验真，以及六个失败点发生时整单回滚。`26 passed` 只证明这些路径在隔离 PostgreSQL 17 fixture 中通过，不证明共享演示库、Docker/API、浏览器或真实模型端到端链路已经通过。

### 4.5 如果分析能力根本不存在，管理员 Agent 可以交出什么？

&emsp;&emsp;服务等级案例展示“能力已经形成”的正向路径。接下来换一个反向问题：管理员提出“我要让分析师能按会员等级看销售额”，但当前系统没有会员等级字段、枚举值和权威数据来源。此时模型不能因为系统里已有 `net_sales` 就自行编造会员等级。

&emsp;&emsp;先区分两条意图链：`DATA_ANALYST` 查询意图识别回答“分析师想查什么”，终点是 `submit_query_plan`；`DATABASE_ADMIN` 变更意图识别回答“管理员想让系统改变什么”，终点只能是下面三个受限提交动作。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 4-3　管理员变更意图的三个终点</font></p>

<div class="courseware-table-wrap">

| 提交动作 | 什么时候使用 | 提交以后发生什么 |
| --- | --- | --- |
| `submit_change_intent` | 目标、对象、规则和数据来源已经足够明确 | 提交结构化变更申请，进入后续审校；不直接执行 SQL |
| `submit_clarification` | 缺少一个会改变方案的关键事实 | 向管理员提出具体问题，本轮停止在澄清状态 |
| `submit_no_change_needed` | 读取当前事实后确认系统已经满足需求 | 说明无需变更的依据，不创建虚假变更 |

</div>

&emsp;&emsp;运行代码时，先观察三个层次：13 个候选 Skills 提供变更方法和规则；11 个只读工具读取 Skill、对象结构、治理视图、语义定义和窄事实；三个 `submit_*` 工具才是本轮允许到达的终点。候选范围由程序收窄，具体选择由模型结合读取到的事实作出。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 4-4　真实模型代码的八个观察步骤</font></p>

<div class="courseware-table-wrap">

| 代码步骤 | 学员要观察什么 | 关键边界 |
| --- | --- | --- |
| 读取候选项 | 13 个 Skills、11 个只读工具和 3 个提交动作 | Skills 是规则正文，不是可执行函数 |
| 打印源码地图 | 五个封装位置及建议搜索符号 | 文件存在不等于业务链路已运行 |
| 准备配置与输入 | 从项目 `.env` 读取模型配置，单独保存管理员原话 | Key 只在内存中使用，不打印值 |
| 记录公开工具 | 只保存工具名和公开参数 | 不展示或推断模型隐藏思维过程 |
| 组装领域上下文 | 当前对象目录、Skill 目录和管理员输入进入 Prompt | 上下文只描述当前可读事实 |
| 创建本轮 Agent | 每次 operation 创建新的管理员 Agent | 不复用上一次 operation 的临时状态 |
| 绑定受限端口并运行 | 只读事实端口与三个终态提交端口被绑定 | 没有 SQL 或数据库执行工具 |
| 打印最终结果 | 工具轨迹、最终提交动作和失败关闭状态 | 普通文本不能冒充结构化终态 |

</div>

### 4.6 三个提交工具怎样经过确定性代码进入应用层？

&emsp;&emsp;模型生成 `submit_*` 工具调用后，不会直接跳到数据库。它还要经过工具注册、参数校验、本轮 Port 和状态解析、应用层分流，以及 Agent 停止条件。按下面五个入口阅读源码，可以看到完整封装链，而不需要从整个项目漫无目的地搜索。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 4-5　三个提交工具的源码阅读地图</font></p>

<div class="courseware-table-wrap">

| 阅读顺序 | 职责 | 源码入口 | 建议搜索符号 |
| --- | --- | --- | --- |
| 1 | 工具注册与分发 | [plugin.py](HermesAnalytics-第三版/backend/src/hermes_analytics/hermes_adapter/admin/plugin.py) | `_dispatch_submit_change_intent`、`install_admin_hermes_plugin` |
| 2 | 参数模型与基础 Handler | [tools.py](HermesAnalytics-第三版/backend/src/hermes_analytics/hermes_adapter/admin/tools.py) | `ChangeIntentInput`、`handle_submit_*` |
| 3 | 提交 Port 与本轮状态 | [port.py](HermesAnalytics-第三版/backend/src/hermes_analytics/hermes_adapter/admin/port.py) | `AdminIntentSubmissionPort`、`AdminOperationState` |
| 4 | 应用层三路分流 | [service.py](HermesAnalytics-第三版/backend/src/hermes_analytics/application/admin_intent/service.py) | `_CapturingIntentPort`、`receive_intent` |
| 5 | 工具白名单与停止条件 | [agent.py](HermesAnalytics-第三版/backend/src/hermes_analytics/hermes_adapter/admin/agent.py) | `_INTENT_TOOLS`、`_post_tool_results_halt_decision` |

</div>

> **记住调用链**：模型生成工具调用 → Plugin 分发 → 参数和 Skill 已读状态校验 → Port 接收本轮提交 → 应用层进入变更、澄清或无需变更分支 → Agent 在终态被接受后停止。

### 4.7 真实模型为什么选择澄清，而不是生成变更？

&emsp;&emsp;当前保存的真实调用以“我要让分析师能按会员等级看销售额”为输入。项目配置请求的模型名是 `deepseek-v4-flash`，本次使用 `RUN_ADMIN_INTENT_LIVE_TESTS=1` 显式开启调用；输出只记录公开工具调用，不记录隐藏思维过程。模型依次完成 17 次工具调用：

```text
 1. read_skill                   -> change-overview
 2. search_objects               -> member / membership / level / vip / 会员 / customer
 3. read_object_schema           -> ecommerce_core.orders
 4. read_object_schema           -> ecommerce_core.order_items
 5. read_object_schema           -> analytics.v_paid_order_items
 6. read_object_schema           -> analytics.v_paid_orders
 7. read_object_schema           -> ecommerce_core.products
 8. read_object_schema           -> ecommerce_core.refund_items
 9. read_object_schema           -> analytics.v_refund_items
10. read_skill                   -> create-dimension
11. read_skill                   -> create-table
12. read_skill                   -> change-governance-view
13. read_semantic_definition     -> metric:net_sales
14. read_view_definition         -> analytics.v_paid_order_items
15. read_view_definition         -> analytics.v_paid_orders
16. read_view_definition         -> analytics.v_paid_order_items
17. submit_clarification         -> 追问会员等级取值及权威数据来源
```

最终输出是：

```text
提交动作： submit_clarification
向管理员追问：系统里目前没有任何会员等级字段或数据来源。要让分析师按会员等级看销售额，需要新增一个会员等级维度。请告知会员等级有哪些具体取值，以及这些等级数据从哪里获取？
选择原因：数据来源或业务规则不足，继续生成会变成猜测。
模型调用：PASS
```

&emsp;&emsp;这条轨迹说明模型先读规则，再读取对象、Schema、治理视图和 `net_sales` 语义定义；确认关键业务事实缺失后，选择澄清而不是编造方案。公开输出中的 `ADMIN_CLARIFICATION_ACCEPTED` 表示终止护栏已经接受 `submit_clarification` 并停止本轮，这是终态被接受，不是调用失败。

&emsp;&emsp;这里的 `PASS` 只表示一次真实调用产生了符合工具契约的结构化终态。它不证明远端返回模型身份已经被独立解析，不证明 SQL 已生成，也不证明审校、freeze、dry-run、数据库执行、Docker 或浏览器链路已经成功。

### 4.8 一次成功变更为什么不能自动变成下一次规则？

&emsp;&emsp;服务等级能力验真解决“现在能不能分析”，会员等级澄清解决“事实不足时能不能提交变更”。一次任务结束后，系统还可能总结哪些修正值得复用，但经验不能越过人类裁决直接影响下一次意图识别。

- 仓库中的 Skills 是**流程区**，说明 Agent 应怎样读取事实、组织申请和处理某类变更；
- 数据库中的活动约定是**认可区**，只保存已经通过治理并允许继续使用的经验；
- `SkillAssemblyService` 在读取时把流程区和活动约定现场拼装，不生成第二份落盘真源；
- Skills 与约定都不是业务表、治理视图或活动领域版本，不是三层数据对象之外的“第四层”。

&emsp;&emsp;可泛化修正只能先形成提案。`PENDING` 等待人类处理，不参与下一次 Skills 拼装；`APPROVED` 才允许新版本进入认可区；`DISCARDED` 终结提案但不写入约定。没有可靠差值时不形成新约定，可以避免把一次偶然修正固化成长期规则。

源码锚点：[Skills 装配](HermesAnalytics-第三版/backend/src/hermes_analytics/application/admin_skills/assembly.py)、[提案服务](HermesAnalytics-第三版/backend/src/hermes_analytics/application/admin_skills/service.py)、[状态枚举](HermesAnalytics-第三版/backend/src/hermes_analytics/domain/skill_convention.py)。

### 4.9 怎样读懂 `PENDING` 状态探针？

&emsp;&emsp;这段代码只读取两套状态枚举，并手工指定一个 `PENDING` 输入。它不连接数据库，不创建提案，不执行批准或丢弃，也不发布新约定。

In [6]:
"""读取 Skills 的两套状态机，展示提案何时真正生效。"""
from hermes_analytics.domain.skill_convention import ConsolidationDecision, ConventionStatus

# 分别读取已生效约定的状态，以及复盘提案等待人工裁决的状态。
convention_statuses = [item.value for item in ConventionStatus]
proposal_decisions = [item.value for item in ConsolidationDecision]

# 本例输入是一条尚未由管理员裁决的提案。
proposal = ConsolidationDecision.PENDING

print('【输入】当前复盘提案')
print('提案状态：', proposal.value)
print('\n【过程 1】读取两套相互独立的状态机')
print('约定区状态：', convention_statuses)
print('提案区裁决状态：', proposal_decisions)
print('\n【过程 2】检查 PENDING 是否已经产生治理效果')
print('当前提案是否已经批准：', proposal is ConsolidationDecision.APPROVED)
print('当前提案是否已经丢弃：', proposal is ConsolidationDecision.DISCARDED)
print('\n【输出】PENDING 只等待人类裁决，不会进入下一轮生效约定。')
print('【本段边界】本探针只读取枚举，不会写入约定区，也不证明提案审批流程已完成。')

【输入】当前复盘提案
提案状态： PENDING

【过程 1】读取两套相互独立的状态机
约定区状态： ['ACTIVE', 'ARCHIVED']
提案区裁决状态： ['PENDING', 'APPROVED', 'DISCARDED']

【过程 2】检查 PENDING 是否已经产生治理效果
当前提案是否已经批准： False
当前提案是否已经丢弃： False

【输出】PENDING 只等待人类裁决，不会进入下一轮生效约定。
【本段边界】本探针只读取枚举，不会写入约定区，也不证明提案审批流程已完成。


当前保存输出为：

```text
【输入】当前复盘提案
提案状态： PENDING

【过程 1】读取两套相互独立的状态机
约定区状态： ['ACTIVE', 'ARCHIVED']
提案区裁决状态： ['PENDING', 'APPROVED', 'DISCARDED']

【过程 2】检查 PENDING 是否已经产生治理效果
当前提案是否已经批准： False
当前提案是否已经丢弃： False

【输出】PENDING 只等待人类裁决，不会进入下一轮生效约定。
【本段边界】本探针只读取枚举，不会写入约定区，也不证明提案审批流程已完成。
```

&emsp;&emsp;`ACTIVE`、`ARCHIVED` 属于已进入认可区的约定状态；`PENDING`、`APPROVED`、`DISCARDED` 属于提案裁决状态。两个 `False` 只说明本例仍在等待裁决，不能证明真实审批接口、数据库写入或发布流程已经运行。

### 4.10 当前证据能证明到哪一层？

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 4-6　第四章证据与外推边界</font></p>

<div class="courseware-table-wrap">

| 证据 | 当前结果 | 可以证明 | 仍不能证明 |
| --- | --- | --- | --- |
| 管理员组件契约 | `175 passed` | 数据模型、三趟审校、编译自检、冻结生命周期与状态机的定向契约通过 | PostgreSQL、部署栈或浏览器主线通过 |
| 隔离 PostgreSQL 17 跨层用例 | `26 passed` | 冻结前提、正反向提交、隐式副作用拒绝、三处验真与失败回滚在隔离 fixture 中通过 | 共享演示库、任意变更类型或完整管理员端到端链路通过 |
| 三个提交工具源码地图 | 5 个相对路径和核心符号存在 | 注册、校验、Port、应用层分流和停止条件可以沿源码定位 | 这些分支已经被真实业务请求全部走过 |
| 管理员意图真实调用 | 17 次公开工具调用；最终 `submit_clarification` | 一次模型调用读取规则与事实后提交了结构化澄清 | 解析模型身份、长期稳定性、SQL、数据库或完整变更链路通过 |
| `PENDING` 状态探针 | 批准与丢弃均为 `False` | 两套枚举及“待裁决不生效”的当前代码表达 | 已创建真实提案、已完成人工审批或已发布约定 |

</div>

### 4.11 完成第四章后，你应该能回答什么？

1. Agent 提交六步 `ChangeIntent` 后，哪四步会执行 SQL，哪两步只形成领域包补丁？
2. 服务等级的回填规则是什么，为什么不会覆盖已有非空值？
3. 新领域版本在什么时候写入并成为活动版本？
4. 新分析会话怎样读取新活动版本，旧 FrozenQuery 为什么保持不变？
5. 四个能力阶段为什么只需要三个验真位置？
6. 会员等级需求为什么必须进入 `submit_clarification`，而不是生成变更申请？
7. 三个 `submit_*` 工具怎样经过确定性代码进入应用层？
8. 为什么 `PENDING` 提案不能影响下一次意图识别？

<details><summary>参考要点</summary>

1. 表结构、业务数据和两份治理视图共四步执行确定性编译得到的 SQL；维度和指标两步只形成领域包补丁，由整单候选 manifest 统一发布。
2. `order_amount < 1000` 写 `L1`，其余写 `L2`；目标过滤限定为 `service_level IS NULL`，所以已有非空值不在本次回填范围内。
3. 六步全部成功后，执行器才校验并发布候选 manifest：写入 `system.domain_versions`，再更新 `active_domain_version_id`；发布或终态核对失败则整单回滚。
4. 新 operation 读取当前活动领域版本、数据修订和 manifest，再用新指标与维度生成 FrozenQuery；旧 Query 继续绑定旧版本坐标，因此结果不被反向改写。
5. 业务表与治理视图都在物理数据库核对，语义发布和实际分析使用分别在活动领域版本与分析师运行态核对。
6. 当前系统缺少会员等级字段、枚举值和权威数据来源，继续生成会把猜测写进治理申请。
7. 工具调用依次经过 Plugin 分发、参数与 Skill 已读校验、Port 接收、应用层三路分流和 Agent 终态停止；没有直接 SQL 执行入口。
8. `PENDING` 只是待人类裁决的提案，不参与 Skills 拼装；只有 `APPROVED` 才可能把新版本写入认可区。

</details>

&emsp;&emsp;完成能力验真后，需要继续守住三条判断边界：能力是否真的跨层生效，缺失能力是否被如实识别，可复用经验是否经过人工批准。服务等级能力通过验真后，分析师还必须真正完成一次分析并留下结论、FactRef（事实引用）与 ResultSnapshot（结果快照）；只有满足完成态、引用和快照门槛的分析，才有资格进入 Evidence 候选筛选。

## <center>第五章：哪些可信分析可以进入归因工作台</center>

&emsp;&emsp;管理员的可信写链先让新能力进入分析运行态，分析师再基于已治理能力完成可信分析。归因工作台接收的不是管理员变更单、原始表或临时查询，而是已经完成、仍可追溯的分析结果；因此从这里开始，课程主线由“怎样获得可信答案”转向“怎样围绕可信答案审阅解释”。

&emsp;&emsp;前四章的服务等级案例到此完成。归因部分改用渠道下降案例，因为这个案例已经具备整体渠道变化、下降最多渠道和重点渠道诊断等多条完成分析。两段案例共享同一条系统契约：分析能力必须先经过治理，分析结果必须再经过完成态与引用复检；只有通过这两道门，结果才可能成为归因 Evidence。

&emsp;&emsp;本章会沿着候选筛选、FactRef 复检、ResultSnapshot 可用性检查和 start 时冻结四步，判断一条完成分析为什么可能仍不能进入证据篮。你还会区分服务器保存的完整 EvidenceSnapshot（证据快照）与模型只看到的最小投影。理解这层入口门槛后，后面的多角色讨论才有共同且不漂移的事实底座。

&emsp;&emsp;归因工作台不是因果推断、精确贡献分解或自动决策系统。它要解决的是：现有证据支持哪些解释、反对哪些解释、还缺哪些证据，并把这个审阅过程保存成可追溯的公开记录。

<div align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164228392.png" alt="完成分析通过候选门槛，在 start 时冻结完整 EvidenceSnapshot，并向 Agent 提供最小投影" width="85%"></div>

### 5.1 新能力生效后，为什么还不能直接开始归因？

&emsp;&emsp;管理员可信变更链的终点是新的分析能力可运行，但“能力可用”还不是归因输入。分析师必须先提出具体问题、完成受控查询并生成带 FactRef 的结论；这次分析的 Turn（分析回合）、Query（查询记录）和 ResultSnapshot 还要满足候选复检条件。

&emsp;&emsp;因此，渠道下降案例进入归因前至少要经历：完成渠道分析 → 保存结论与 FactRef → 核对 ResultSnapshot 可用性 → 进入证据篮候选 → 点击 start 时冻结 EvidenceSnapshot。归因运行不会绕过这条读链重新查询数据库，也不会把后台管理员的变更记录当成业务证据。

### 5.2 为什么归因只能使用启动时冻结的 Evidence？

&emsp;&emsp;误解是“分析完成后，运行时会随时读取最新分析结果”。真实机制是：归因只围绕 start 时选定并冻结的 EvidenceSnapshot 工作。完整 FactRef、指标定义、结果快照 ID 和来源坐标保留在冻结层；模型上下文只接收最小投影：`evidence_id + analyst_conclusion`。

&emsp;&emsp;用户填写的背景始终带 `USER_PROVIDED_UNVERIFIED`。它可以帮助理解问题，却不能升级为数据事实，更不能替代冻结 Evidence。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 5-1　归因阶段的数据可见范围</font></p>

<div class="courseware-table-wrap">

| 层 | 归因阶段可用内容 | 不应误读为 |
| --- | --- | --- |
| 已完成分析 | 候选来源 | 自动进入归因的任何结果 |
| EvidenceSnapshot | FactRef、指标定义、快照与冻结时间 | 全部塞给模型的上下文 |
| Hermes Agent 上下文 | `evidence_id`、`analyst_conclusion`、允许的公开历史 | 新的数据库查询出口 |
| 用户背景 | 内容 + `USER_PROVIDED_UNVERIFIED` | 可信业务事实 |

</div>

### 5.3 哪些完成态分析可以进入证据篮？

&emsp;&emsp;`COMPLETED` 只是第一道门，不是充分条件。页面的证据篮只展示同一 owner 下同时通过下表校验的分析；任一条件不满足，就会被排除，不能在 start 时冻结。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 5-2　完成态分析进入证据篮的条件</font></p>

<div class="courseware-table-wrap">

| 判定 | 可进入证据篮 | 典型被排除原因 |
| --- | --- | --- |
| 来源归属与终态 | 同一 owner 的 Turn，且为 `COMPLETED`，outcome 是 `SUCCEEDED` 或 `PARTIALLY_SUCCEEDED` | owner 不同、仍在运行、`FAILED` / `CANCELLED`，或 outcome 不在这两个值中 |
| 来源会话与结论 | 来源分析会话未归档；assistant conclusion 非空，且 FactRef 列表非空 | 会话已归档、结论为空、没有 FactRef |
| FactRef 指向的 Query | 每条 FactRef 对应的 Query 都是 `SUCCEEDED` 或 `RESULT_LIMITED` | Query 缺失、失败、取消或不是允许终态 |
| ResultSnapshot 与坐标 | 对应 ResultSnapshot 的 trust 为 `ANALYSIS_VERIFIED`、payload 为 `AVAILABLE`；FactRef 的 snapshot ID 与快照中的 `fact_ref_id`、`query_id`、`result_snapshot_id`、`row_key`、`column_key` 坐标匹配 | 快照不可用或未验证、snapshot ID 不一致、坐标在快照 FactRef 中找不到、FactRef 重复或格式不合法 |

</div>

&emsp;&emsp;你可以把它理解成“完成分析的可追溯性复检”：证据篮接受的不是一条完成记录，而是一组仍可核验的结论、引用和结果快照。源码锚点：`attribution_repository.py` 的 `list_evidence_candidates()`、`_candidate_for_turn()`、`_fact_ref()`。

&emsp;&emsp;这里还要拆掉一个很容易被状态名称误导的理解：**`SUCCEEDED` 只说明查询技术链走到了成功终态，不自动证明模型理解的业务问题正确。**一条分析即使通过了候选的技术校验，仍可能在冻结 Intent 中漏掉应有的分组、比较、过滤或排名。例如，问题要求“按渠道比较净销售额环比”，至少应看到渠道分组、上一期比较和正确时间范围；问题要求“抖音渠道具体发生了什么”，诊断 Query 还应保留抖音过滤条件。缺少这些业务语义时，这条分析不能作为正式归因 Evidence。

&emsp;&emsp;一次 2026-08-26 的独立环境复测出现过“四轮技术状态均成功，但冻结 Intent 缺少关键分组、排名或过滤”的反例。这个样本说明系统状态与业务语义必须分开验收；它只绑定当时的源码、数据和会话，不能因为页面显示成功就把该会话放进正式归因证据篮。

&emsp;&emsp;源码锚点：`application/attribution/service.py`、`domain/attribution.py`、`hermes_adapter/attribution_context.py`。

### 5.4 学习检查

1. 为什么服务等级能力已经生效，管理员变更单仍不能直接成为归因 Evidence？
2. 一条 Turn 已经 `COMPLETED`，为什么仍可能进不了证据篮，或者虽然能进入候选却不适合作为正式归因 Evidence？
3. start 时冻结 EvidenceSnapshot 解决了什么问题？模型是否因此获得完整 FactRef？

<details><summary>参考要点</summary>

1. 管理员变更只证明分析能力可用；归因输入必须是分析师实际完成、带结论与可追溯引用的可信分析。
2. 系统还要检查 owner、outcome、会话状态、assistant conclusion、FactRef、Query 终态、ResultSnapshot trust/payload 以及引用坐标是否一致；选入正式 Evidence 前还要人工核对冻结 Intent 的指标、分组、比较、过滤和排名是否真的回答原问题。
3. 冻结让本次归因始终围绕同一批证据，不随源分析继续变化；完整 FactRef 留在服务器冻结层，模型只接收最小投影。

</details>

&emsp;&emsp;确认“哪些证据可以进场”之后，还要回答“同一批冻结 Evidence 应该交给哪些受限角色，以及每个角色究竟能做什么”。

## <center>第六章：Hermes Agent 如何承担三个归因角色</center>

&emsp;&emsp;第五章把符合条件的已完成分析选出来，并在归因运行启动时冻结成 Evidence 快照。冻结动作发生在三个归因角色开始工作之前；ADVOCATE（支持方）、SKEPTIC（质疑方）、JUDGE（裁决方）不负责再次冻结证据，它们只接收这份快照的最小投影和当前允许公开的讨论历史。

&emsp;&emsp;本章只回答“谁可以围绕同一批 Evidence 做什么”：先区分 Agent、Factory、Adapter 与应用层 Handler 的责任，再认识三个角色的输入、动作和禁止项，最后用四段本地代码依次核对类职责、角色登记、Prompt 指纹与工具隔离。角色真实调用的先后顺序留到第七章验证。

### 6.1 谁编排完整流程，谁只执行一次角色调用？

&emsp;&emsp;这里有四层代码共同工作，但它们掌握的控制权不同。最容易混淆的是：`HermesAttributionAdapter` 能调用角色 Agent，不代表它能决定完整归因流程；真正决定角色数量、并行屏障、轮次上限和报告时机的是应用层 `AttributionOperationHandler`。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 6-1　四层代码的责任边界</font></p>

<div class="courseware-table-wrap">

| 层 | 负责什么 | 不负责什么 |
| --- | --- | --- |
| `HermesAnalyticsAgent` | 运行一次受限模型调用，锁定系统 Prompt、显式工具白名单、上下文和迭代边界 | 不决定有几个角色，也不自由读取记忆、文件或其他工具 |
| `HermesAgentFactory` | 每次调用创建一个新 Agent，绑定当前角色唯一的 `enabled_toolsets`，并核对实际工具集合 | 不决定角色调用顺序和归因轮数 |
| `HermesAttributionAdapter` | 一次运行一个 ADVOCATE（支持方）、SKEPTIC（质疑方）或 JUDGE（裁决方），并校验角色、阶段、轮次与结构化提交 | 不编排完整的多轮讨论 |
| `AttributionOperationHandler` | 在应用层决定固定角色、调用顺序、并行屏障、轮次上限和报告时机 | 不允许模型新增角色、改变拓扑或绕过门禁 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164218327.png" alt="应用层编排与单角色 Hermes Agent 执行的责任边界" width="85%"></div>

&emsp;&emsp;可以把它理解成一条受控装配线：Handler 是控制器，决定哪个工位何时启动、何时汇合、何时进入裁决；Factory 按当前工位创建一个全新的角色 Agent，并发放专属工具箱；Adapter 只负责完成当前这一次角色调用。这个类比只解释职责和调用关系，不表示 Agent 可以像人一样协商流程。

&emsp;&emsp;源码锚点：[agent.py](HermesAnalytics-第三版/backend/src/hermes_analytics/hermes_adapter/agent.py)、[attribution.py](HermesAnalytics-第三版/backend/src/hermes_analytics/hermes_adapter/attribution.py)、[operation.py](HermesAnalytics-第三版/backend/src/hermes_analytics/application/attribution/operation.py)。

### 6.2 为什么每次都要创建新的单角色 Agent？

- “多 Agent 框架会自动分工”不成立。固定角色和调用拓扑写在应用层 Handler 中。
- “一个 Agent 可以顺手调用所有工具”不成立。Factory 每次只传入当前角色唯一的 `enabled_toolsets`，并核对实际工具名是否与源码合同完全一致。
- “同一个 Agent 会带着隐藏记忆跨轮工作”不成立。每次开场、反驳或裁决调用都会创建新的角色 Agent，历史只能由应用层按规则重新注入。

&emsp;&emsp;这样做的目的不是把 Agent 数量做多，而是把一次调用的身份、Prompt、工具和上下文同时锁定。上一轮未公开的内容不会因为复用同一实例而悄悄进入下一轮；真正允许共享的内容，必须先成为应用层可见的公开结构化记录。

### 6.3 ADVOCATE（支持方）、SKEPTIC（质疑方）、JUDGE（裁决方）各自能做什么？

&emsp;&emsp;三个角色面对的是同一份已冻结 Evidence，但任务并不相同。支持方提出证据能够支撑的最强解释，质疑方检查解释中的跳跃和缺口，裁决方判断分歧是否还值得继续；它们不是三个投票者，也不是三个可以自由查询数据的分析师。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 6-2　三个归因角色的输入、动作与正式输出</font></p>

<div class="courseware-table-wrap">

| 角色 | 输入 | 核心职责 | 允许动作与专属工具 | 正式输出 |
| --- | --- | --- | --- | --- |
| ADVOCATE（支持方） | Evidence 最小投影、允许公开的历史、角色与轮次上下文 | 提出当前证据能支持的解释，区分数值贡献、相关关系和因果机制，并暴露假设 | `POSITION`；`publish_attribution_advocate_position` | 支持方公开立场 |
| SKEPTIC（质疑方） | Evidence 最小投影、允许公开的历史、可引用的 Evidence 与 Claim 白名单 | 检查证据支持、因果跳跃、替代解释和证据缺口；证据充分时应当让步 | `POSITION`；`publish_attribution_skeptic_position` | 质疑方公开审查 |
| JUDGE（裁决方） | Evidence 最小投影、双方公开记录、当前轮次与 `allow_next_round` 门禁 | 按证据判断继续一轮还是形成报告，不按意见数量投票 | `NEXT_ROUND` / `REPORT`；`request_next_attribution_round` / `publish_attribution_report` | 下一轮请求或最终报告，二选一 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164218410.png" alt="三角色的输入、职责、专属工具与公开输出" width="85%"></div>

&emsp;&emsp;ADVOCATE（支持方）是“证据范围内的最强解释者”，不是因果证明者。它不能因为某个因素贡献高，就直接宣称该因素已经被证明是原因；新证据或有效反驳出现后，它还应修正或撤回主张。

&emsp;&emsp;SKEPTIC（质疑方）是“证据审查者”，不是为了反对而反对。它引用的 `evidence_id` 与 `target_claim_id` 必须来自当前白名单，不能凭空创造证据或攻击不存在的主张。

&emsp;&emsp;JUDGE（裁决方）是“证据充分性裁决者”，不是投票箱。只有应用层允许继续时，它的 `NEXT_ROUND` 请求才可能被接受；达到轮次上限或继续门禁关闭后，只能提交 `REPORT`。一次裁决调用也只能留下一个有效出口。

&emsp;&emsp;三个角色共同受以下边界约束：只能使用本次启动时已冻结的 Evidence 与允许公开的历史；用户补充的背景仍是 `USER_PROVIDED_UNVERIFIED`，不能升级为数据事实；不能访问数据库、SQL、终端、文件、浏览器、代码执行器或外部网络；普通文本 final 不进入正式归因记录，只有获准的结构化工具提交经过校验后才算业务动作。

&emsp;&emsp;约束分为三层：Prompt 说明职责与禁止事项，运行时只暴露当前角色的工具集，服务端再校验动作、Evidence/Claim 白名单、轮次与结构。这里先观察前两层，服务端白名单与可信字段补全则在模型提交边界中单独核对。源码锚点：[attribution_prompt.py](HermesAnalytics-第三版/backend/src/hermes_analytics/hermes_adapter/attribution_prompt.py)、[plugin.py](HermesAnalytics-第三版/backend/src/hermes_analytics/hermes_adapter/plugin.py)。

### 6.4 怎样用四段代码逐层核对角色合同？

&emsp;&emsp;进入第三版源码的 `backend` 目录（相对课件目录为 `HermesAnalytics-第三版/backend`）后，按顺序运行下面四段。代码必须使用系统枚举名 `ADVOCATE`、`SKEPTIC`、`JUDGE`；它们分别对应中文的支持方、质疑方和裁决方。四段代码研究的是本地类职责、Prompt 与工具注册表，不连接数据库，也不调用真实模型。下面展示的是 Notebook 已保存的既有运行结果；阅读这些结果不等于本次重新运行。

#### 6.4.1 第一段：先确认三层基座各管什么

In [7]:
"""直接读取三个核心类的源码说明，定位 Hermes Agent 的责任边界。"""
from hermes_analytics.hermes_adapter.agent import HermesAgentFactory, HermesAnalyticsAgent
from hermes_analytics.hermes_adapter.attribution import HermesAttributionAdapter

base_contracts = (
    ('HermesAnalyticsAgent', HermesAnalyticsAgent.__doc__),              # 锁定上下文和单次迭代边界。
    ('HermesAgentFactory', HermesAgentFactory.__doc__),                  # 每次调用创建新 Agent 并绑定工具集。
    ('HermesAttributionAdapter', HermesAttributionAdapter.__doc__),      # 一次只运行一个归因角色。
)
print('【输出】Hermes Agent 三层责任')
for component, contract in base_contracts:
    print(f'- {component}：{(contract or "").strip()}')
print('结论：Agent 负责单角色调用；Handler 负责角色数量、先后顺序和轮数。')

【输出】Hermes Agent 三层责任
- HermesAnalyticsAgent：真正替换 Hermes 通用身份，并锁死上下文与迭代边界。
- HermesAgentFactory：每次 operation 创建全新 Agent，并按阶段锁死工具集合。
- HermesAttributionAdapter：一次只运行一个归因角色；多轮顺序由应用层 Handler 决定。
结论：Agent 负责单角色调用；Handler 负责角色数量、先后顺序和轮数。


```text
【输出】Hermes Agent 三层责任
- HermesAnalyticsAgent：真正替换 Hermes 通用身份，并锁死上下文与迭代边界。
- HermesAgentFactory：每次 operation 创建全新 Agent，并按阶段锁死工具集合。
- HermesAttributionAdapter：一次只运行一个归因角色；多轮顺序由应用层 Handler 决定。
结论：Agent 负责单角色调用；Handler 负责角色数量、先后顺序和轮数。
```

&emsp;&emsp;这段输出证明三个类在源码中的职责声明彼此分开。它没有读取 Handler 的实际运行轨迹，因此只回答“基座设计成谁负责什么”，不证明完整流程已经执行。

#### 6.4.2 第二段：准备角色登记与运行时工具读取函数

In [8]:
"""准备三个角色的职责、Prompt Composer 与运行时工具读取函数。"""
import io
from contextlib import redirect_stderr, redirect_stdout

from model_tools import get_tool_definitions

from hermes_analytics.hermes_adapter.attribution_prompt import AttributionPromptComposer
from hermes_analytics.hermes_adapter.plugin import (
    ATTRIBUTION_ADVOCATE_TOOLSET,
    ATTRIBUTION_JUDGE_TOOLSET,
    ATTRIBUTION_SKEPTIC_TOOLSET,
    install_hermes_plugin,
)

def registered_tools(toolset: str) -> list[str]:
    """返回一个角色工具集中已经注册的公开工具名。"""
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        definitions = get_tool_definitions(
            enabled_toolsets=[toolset],                                # 只启用当前角色的唯一工具集。
            quiet_mode=True,                                           # 不把注册提示混入学习输出。
        )
    return sorted(item['function']['name'] for item in definitions)     # 只保留最需要观察的工具名。


with redirect_stderr(io.StringIO()):
    install_hermes_plugin()                                             # 幂等安装本项目的工具定义。

prompt_composer = AttributionPromptComposer()                           # 组合三个角色的稳定系统 Prompt。
role_contracts = (
    ('ADVOCATE', '提出当前证据能够支持的解释', ATTRIBUTION_ADVOCATE_TOOLSET),
    ('SKEPTIC', '检验主张、替代解释与证据缺口', ATTRIBUTION_SKEPTIC_TOOLSET),
    ('JUDGE', '按证据决定继续一轮还是生成报告', ATTRIBUTION_JUDGE_TOOLSET),
)
expected_tools = {
    'ADVOCATE': ['publish_attribution_advocate_position'],
    'SKEPTIC': ['publish_attribution_skeptic_position'],
    'JUDGE': ['publish_attribution_report', 'request_next_attribution_round'],
}
print('【输入】角色数量：', len(role_contracts))
print('【输入】角色顺序：', [role for role, _, _ in role_contracts])

【输入】角色数量： 3
【输入】角色顺序： ['ADVOCATE', 'SKEPTIC', 'JUDGE']


```text
【输入】角色数量： 3
【输入】角色顺序： ['ADVOCATE', 'SKEPTIC', 'JUDGE']
```

&emsp;&emsp;这里先安装本地工具定义，再准备三份角色合同。输出中的“角色顺序”只是 `role_contracts` 元组的展示与遍历顺序，不是完整流程的真实调用顺序；开场是否并行、Judge 何时后置，必须以应用层实际运行轨迹为准。

#### 6.4.3 第三段：检查 Prompt 版本和内容指纹

In [9]:
"""逐角色打印职责、允许动作以及 Prompt 版本指纹。"""
print('【输出】三个角色的 Prompt 合同')
for role, duty, _ in role_contracts:
    prompt = prompt_composer.compose(role=role)                          # 从本地源码生成当前角色 Prompt。
    actions = (
        ['POSITION']
        if role != 'JUDGE'
        else ['NEXT_ROUND', 'REPORT']
    )                                                                    # Judge 才能结束或继续流程。

    print()
    print(f'角色：{role}')
    print('职责：', duty)
    print('允许动作：', actions)
    print('Prompt 版本：', prompt.prompt_version)
    print('Prompt 哈希：', prompt.prompt_hash[:23] + '...')              # 保留 sha256 前缀和短指纹，减少视觉噪声。
    print('是否为版本化哈希：', prompt.prompt_hash.startswith('sha256:'))

【输出】三个角色的 Prompt 合同

角色：ADVOCATE
职责： 提出当前证据能够支持的解释
允许动作： ['POSITION']
Prompt 版本： attribution-v2:ADVOCATE
Prompt 哈希： sha256:8ac792f05bb3d0b5...
是否为版本化哈希： True

角色：SKEPTIC
职责： 检验主张、替代解释与证据缺口
允许动作： ['POSITION']
Prompt 版本： attribution-v2:SKEPTIC
Prompt 哈希： sha256:dfec953e2793729e...
是否为版本化哈希： True

角色：JUDGE
职责： 按证据决定继续一轮还是生成报告
允许动作： ['NEXT_ROUND', 'REPORT']
Prompt 版本： attribution-v2:JUDGE
Prompt 哈希： sha256:435999c6bf58889d...
是否为版本化哈希： True


```text
角色：ADVOCATE
允许动作： ['POSITION']
Prompt 版本： attribution-v2:ADVOCATE
Prompt 哈希： sha256:8ac792f05bb3d0b5...
是否为版本化哈希： True

角色：SKEPTIC
允许动作： ['POSITION']
Prompt 版本： attribution-v2:SKEPTIC
Prompt 哈希： sha256:dfec953e2793729e...
是否为版本化哈希： True

角色：JUDGE
允许动作： ['NEXT_ROUND', 'REPORT']
Prompt 版本： attribution-v2:JUDGE
Prompt 哈希： sha256:435999c6bf58889d...
是否为版本化哈希： True
```

&emsp;&emsp;版本号回答“这次调用采用哪一套角色合同”，哈希把运行记录绑定到那份 Prompt 的具体内容。两次结果不同时，可以继续区分 Evidence、公开历史或 Prompt 是否发生变化。不过，哈希只是内容指纹：它不评价 Prompt 质量，也不证明模型一定遵守 Prompt。

#### 6.4.4 第四段：核对专属工具是否真的隔离

In [10]:
"""打印三个角色的专属工具，并检查禁止能力是否混入。"""
forbidden_tools = {
    'delegate_task', 'terminal', 'read_file',
    'browser_navigate', 'execute_code', 'submit_query_plan',
}                                                                        # 三个归因角色共同禁止的能力。
all_role_tools = set()

print('【输出】角色专属工具')
for role, _, toolset in role_contracts:
    tools = registered_tools(toolset)                                    # 读取运行时真实注册结果。
    all_role_tools.update(tools)
    print(f'- {role}: {tools}')
    print('  与源码合同一致：', tools == expected_tools[role])

print()
print('【输出】工具隔离检查')
print('实际注册工具总数：', len(all_role_tools))
print('禁止能力交集：', sorted(all_role_tools & forbidden_tools))
print('禁止能力是否完全隔离：', all_role_tools.isdisjoint(forbidden_tools))
print('三个角色是否使用三个独立工具集：', len({item[2] for item in role_contracts}) == 3)
print('【本段边界】本地配置通过不等于真实模型一定稳定遵守 Prompt。')

【输出】角色专属工具
- ADVOCATE: ['publish_attribution_advocate_position']
  与源码合同一致： True
- SKEPTIC: ['publish_attribution_skeptic_position']
  与源码合同一致： True
- JUDGE: ['publish_attribution_report', 'request_next_attribution_round']
  与源码合同一致： True

【输出】工具隔离检查
实际注册工具总数： 4
禁止能力交集： []
禁止能力是否完全隔离： True
三个角色是否使用三个独立工具集： True
【本段边界】本地配置通过不等于真实模型一定稳定遵守 Prompt。


```text
【输出】角色专属工具
- ADVOCATE: ['publish_attribution_advocate_position']
  与源码合同一致： True
- SKEPTIC: ['publish_attribution_skeptic_position']
  与源码合同一致： True
- JUDGE: ['publish_attribution_report', 'request_next_attribution_round']
  与源码合同一致： True

【输出】工具隔离检查
实际注册工具总数： 4
禁止能力交集： []
禁止能力是否完全隔离： True
三个角色是否使用三个独立工具集： True
【本段边界】本地配置通过不等于真实模型一定稳定遵守 Prompt。
```

&emsp;&emsp;三个 `True` 分别说明：当前运行时读取到的角色工具与源码期望一致、列出的共同禁止能力没有混入角色工具、三个角色没有共用同一个工具集。工具总数为 4，是因为 ADVOCATE（支持方）与 SKEPTIC（质疑方）各有 1 个提交工具，JUDGE（裁决方）有“请求下一轮”和“发布报告”2 个出口。

&emsp;&emsp;这些结果能够证明当前 Notebook 保存的那次本地运行看到了三套独立工具集和四个预期工具；不能证明 Provider（模型服务提供方）可用、真实模型稳定遵守 Prompt、服务端一定接受提交，也不能证明完整多轮归因或最终报告已经成功。

### 6.5 学习检查

1. Evidence 是由三个角色冻结的吗？三个角色实际接收什么？
2. `HermesAnalyticsAgent`、Factory、Adapter 与 Handler 的控制权分别到哪里为止？
3. 为什么开场、反驳和裁决都要创建新的单角色 Agent？
4. ADVOCATE（支持方）、SKEPTIC（质疑方）、JUDGE（裁决方）的正式动作和专属工具分别是什么？
5. Prompt 哈希与本地工具隔离全部正常后，为什么仍不能声称真实归因已经通过？

<details><summary>参考要点</summary>

1. Evidence 在归因运行启动时已经冻结；角色只接收其最小投影和允许公开的历史，不负责重新冻结或查询数据。
2. Agent 运行一次受限调用，Factory 创建新 Agent 并绑定工具，Adapter 执行一个角色，Handler 决定完整调用拓扑与轮次。
3. 新建实例可以同时锁定身份、Prompt、工具和上下文；跨轮可见信息必须由应用层按规则重新注入，不能依赖隐藏记忆。
4. ADVOCATE（支持方）与 SKEPTIC（质疑方）都只能提交 `POSITION`，各有一个位置发布工具；JUDGE（裁决方）只能在 `NEXT_ROUND` 与 `REPORT` 之间按门禁选择，对应两个专属工具。
5. 本地代码只核对类声明、Prompt 指纹和工具注册；Provider、模型行为、服务端校验与完整流程都需要其他证据。

</details>

&emsp;&emsp;角色合同锁定了“谁能做什么”，但登记顺序不等于执行顺序。应用层还必须让 ADVOCATE（支持方）与 SKEPTIC（质疑方）独立开场、在同轮越过并行屏障，并让 JUDGE（裁决方）始终后置决定继续或结束。

## <center>第七章：多智能体归因如何按固定顺序运行</center>

&emsp;&emsp;角色职责明确后，还要回答另一个问题：这些角色由谁按照什么顺序调用，同一轮看见哪些公开信息，又在什么条件下继续或结束。

&emsp;&emsp;这里的“多智能体”不是三个 Agent（智能体）自由聊天。`AttributionOperationHandler`（归因流程处理器）预先固定调用拓扑；每次轮到某个角色时，Factory（智能体工厂）才创建一个新的单角色 Agent。本章使用真实 Handler 和确定性测试工装，完整展示“独立开场 → 第一轮反驳与裁决 → 第二轮反驳与裁决 → 最终报告”的 8 次调用。

### 7.1 为什么支持方与质疑方要配对运行，裁决方必须后置？

&emsp;&emsp;先认识三个贯穿本章的术语：A/S Pair 指 ADVOCATE（支持方）与 SKEPTIC（质疑方）的一次配对调用；history（公开历史）是当前角色获准看到的既有公开记录；barrier（屏障）表示双方都结束并完成成对发布之前，流程不能进入裁决阶段。

&emsp;&emsp;`AttributionOperationHandler` 的 `_run_position_pair()` 会按八步完成一次 A/S Pair：

1. 从已公开输出生成一份 `history_before_pair`（本次配对开始前的公开历史快照）；
2. 从这份历史提取当前允许引用的 Claim ID（主张编号）；
3. 使用同一份 history、Evidence（冻结证据）白名单和 Claim（公开主张）白名单创建两个 invocation（单次调用请求）；
4. 先登记 Pair 已开始，再启动两个角色任务；
5. 使用两个 `asyncio.create_task()` 并行运行支持方与质疑方；
6. 使用 `asyncio.gather(..., return_exceptions=True)` 等待双方 settled（均已结束），并统一处理取消或失败；
7. 只有双方都返回合法 `POSITION`（角色立场），才装配两条公开输出；
8. 调用 `publish_position_pair()` 成对发布，随后才允许进入下一阶段或调用裁决方。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164218379.png" alt="独立开场、并行屏障与两轮裁决时序" width="85%"></div>

&emsp;&emsp;图解：独立开场先完成一次 A/S Pair；每轮反驳再完成一次 A/S Pair；双方越过成对发布屏障后，JUDGE（裁决方）才读取完整历史并决定 `NEXT_ROUND`（继续下一轮）或 `REPORT`（生成报告）。

&emsp;&emsp;可以把 Pair 理解成两名评审同时拿到密封的同版材料：两人都交卷后，系统才一起公开结果，再把完整材料交给裁决者。这个类比只解释“同版输入、等待双方、成对公开”，不证明数据库写入已经具备跨事务原子回滚。

&emsp;&emsp;事实边界：双方返回先后可以变化，但公开顺序不是“谁先完成谁先发布”；同轮支持方的输出不会临时流入同轮质疑方。裁决方不是第三个并行任务，也不是按票数决定谁赢。源码锚点：[operation.py](HermesAnalytics-第三版/backend/src/hermes_analytics/application/attribution/operation.py)。

### 7.2 为什么“最多两轮”不等于固定 8 次调用？

&emsp;&emsp;`max_rounds`（最大轮数）只是上限，不是必须跑满的数量。独立开场固定调用支持方和质疑方各一次，共 2 次；每个实际完成轮次再调用支持方、质疑方、裁决方各一次，共 3 次。若最大轮数记为 `N`、实际完成轮数记为 `R`，则 `R ≤ N`，真实调用数是 `2 + 3R`，调用上限才是 `2 + 3N`。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 7-1　最大轮数、实际轮数与合法调用数量</font></p>

<div class="courseware-table-wrap">

| 配置的最大轮数 `N` | 可能的实际完成轮数 `R` | 合法调用数量 `2 + 3R` | 结束含义 |
| ---: | --- | --- | --- |
| 1 | 1 | 5 | 第一轮裁决方必须形成报告 |
| 2 | 1 或 2 | 5 或 8 | 第一轮提前收敛，或完整执行两轮 |
| 3 | 1、2 或 3 | 5、8 或 11 | 裁决方可以在非末轮提前形成报告 |

</div>

&emsp;&emsp;裁决方在非末轮既可以提交 `NEXT_ROUND`（继续下一轮），也可以提交 `REPORT`（生成报告）并提前结束。最后一轮的 `allow_next_round`（是否允许继续下一轮）为 `False`，因此不能创建超出上限的新一轮。

&emsp;&emsp;本章工装明确设置两次裁决动作：第一轮 `NEXT_ROUND`，第二轮 `REPORT`。因此本次实际轨迹固定为 `2 + 3 × 2 = 8` 次；表 7-1 中的 5 次路径只是同一 Handler 合同允许的提前结束情况，不是下面代码的本次输出。

### 7.3 怎样用三段代码看清完整的八次调用？

&emsp;&emsp;下面三段代码需要在 Jupyter Notebook（交互式代码课件）中依次运行。第一段准备两轮工装，第二段展开八次调用及 Pair（配对调用）和 Judge（裁决方）细节，第三段集中打印拓扑检查。代码使用 `fixture`（测试夹具）中的确定性 Agent 和 Fake Repository（内存替身仓储），但真正调用的是源码中的 `AttributionOperationHandler`。

&emsp;&emsp;运行时应在课程 Notebook 中按顺序执行；`PROJECT_ROOT` 已由前置公共单元初始化。这里的顶层 `await` 是 Notebook 语法，不应直接复制到普通 `.py` 文件中运行。

#### 7.3.1 第一段：准备完整两轮工装

In [11]:
"""运行真实 Handler 的两轮确定性工装，保存公开调用记录。"""
import sys
BACKEND_ROOT = PROJECT_ROOT / 'backend'                                  # 当前项目的 backend 目录。
sys.modules.pop('tests', None)                                           # 避免其他同名 tests 包遮蔽项目测试。
backend_text = str(BACKEND_ROOT)
if backend_text in sys.path:
    sys.path.remove(backend_text)
sys.path.insert(0, backend_text)                                         # 优先加载当前项目测试 helper。
from tests.unit.test_attribution_operation import _handler, _operation, _topology

MAX_ROUNDS = 2                                                           # 本例运行两轮反驳与裁决。
JUDGE_ACTIONS = ['NEXT_ROUND', 'REPORT']                                 # 第一轮继续，第二轮生成报告。
EXPECTED_CALLS = 2 + 3 * MAX_ROUNDS                                     # 开场 2 次，每轮再增加 3 次。
handler, attribution_repository, deterministic_agent = _handler(
    max_rounds=MAX_ROUNDS,                                               # Handler 的轮数上限。
    judge_actions=JUDGE_ACTIONS,                                         # 确定性 Judge 的两次公开动作。
)
print('【输入】最大轮数：', MAX_ROUNDS)
print('【输入】Judge 动作：', JUDGE_ACTIONS)
print('【输入】预期调用次数：', EXPECTED_CALLS)
await handler.handle(_operation())                                      # 由真实 Handler 决定并行、屏障和 Judge 后置。
attribution_calls = deterministic_agent.calls                           # 保存完成后的公开调用记录。

【输入】最大轮数： 2
【输入】Judge 动作： ['NEXT_ROUND', 'REPORT']
【输入】预期调用次数： 8


```text
【输入】最大轮数： 2
【输入】Judge 动作： ['NEXT_ROUND', 'REPORT']
【输入】预期调用次数： 8
```

&emsp;&emsp;这段代码先把本次情景钉死：最多两轮；第一次 JUDGE（裁决方）选择继续，第二次选择报告。因此 `EXPECTED_CALLS`（预期调用次数）是 8。`handler.handle()` 只调用一次，后面的角色顺序全部由 Handler 内部展开，不是 Notebook 手工逐个调用角色。

#### 7.3.2 第二段：展开 A/S Pair 与后置裁决

In [12]:
"""先打印调用时间线，再展开 _run_position_pair() 的公开过程。"""
import inspect

print('【过程】两轮归因角色时间线')
print('序号 | 角色       | 阶段       | 轮次 | history | next')
print('-' * 64)
for index, call in enumerate(attribution_calls, start=1):
    round_label = '-' if call.round_number is None else str(call.round_number) # 开场阶段没有轮次编号。
    print(
        f'{index:>4} | {call.role:<10} | {call.phase:<10} | '
        f'{round_label:^4} | {len(call.context["debate_history"]):>7} | '
        f'{call.allow_next_round}'
    )
print('说明：前两次是独立开场；每轮双方完成后，Judge 才进入。')

【过程】两轮归因角色时间线
序号 | 角色       | 阶段       | 轮次 | history | next
----------------------------------------------------------------
   1 | ADVOCATE   | OPENING    |  -   |       0 | False
   2 | SKEPTIC    | OPENING    |  -   |       0 | False
   3 | ADVOCATE   | REBUTTAL   |  1   |       2 | False
   4 | SKEPTIC    | REBUTTAL   |  1   |       2 | False
   5 | JUDGE      | JUDGMENT   |  1   |       4 | True
   6 | ADVOCATE   | REBUTTAL   |  2   |       5 | False
   7 | SKEPTIC    | REBUTTAL   |  2   |       5 | False
   8 | JUDGE      | JUDGMENT   |  2   |       7 | False
说明：前两次是独立开场；每轮双方完成后，Judge 才进入。


&emsp;&emsp;先用这段时间线确认八次调用的角色、阶段、轮次和可见历史数量。顺序确认后，再展开每次 A/S Pair（支持方与质疑方配对调用）怎样共享输入、越过屏障并把完整历史交给裁决方。

In [13]:
def public_history_ids(call) -> list[str]:
    """返回当前调用开始时可见的公开 output_id。"""
    return [item['output_id'] for item in call.context['debate_history']]


# 每项绑定 A/S 调用位置；两个反驳阶段再绑定后置 Judge 与工装预设动作。
position_pair_indexes = (
    ('独立开场', 0, 1, None, None),
    ('阶段一：第 1 轮反驳与裁决', 2, 3, 4, JUDGE_ACTIONS[0]),
    ('阶段二：第 2 轮反驳与裁决', 5, 6, 7, JUDGE_ACTIONS[1]),
)

position_pairs = []
judge_calls = []

print('\n【过程】A/S pair 与后置 Judge 的完整流程')
for label, advocate_index, skeptic_index, judge_index, judge_action in position_pair_indexes:
    advocate_call = attribution_calls[advocate_index]
    skeptic_call = attribution_calls[skeptic_index]
    history_ids = public_history_ids(advocate_call)

    pair_outputs = [
        output
        for output in attribution_repository.outputs
        if output.agent_role in {'ADVOCATE', 'SKEPTIC'}
        and output.phase == advocate_call.phase
        and output.round_number == advocate_call.round_number
    ]
    position_pairs.append((advocate_call, skeptic_call, pair_outputs))

    print(f'\n阶段：{label}')
    print('1. 冻结公开历史 ID：', history_ids or '[]（独立开场）')
    print('2. 创建角色 invocation：', [advocate_call.role, skeptic_call.role])
    print(
        '3. 两侧 history 完全相同：',
        advocate_call.context['debate_history'] == skeptic_call.context['debate_history'],
    )
    print(
        '4. Evidence 白名单：',
        f'A={len(advocate_call.allowed_evidence_ids)} 条，',
        f'S={len(skeptic_call.allowed_evidence_ids)} 条，',
        f'完全相同={advocate_call.allowed_evidence_ids == skeptic_call.allowed_evidence_ids}',
    )
    print(
        '5. Claim 白名单：',
        f'A={sorted(advocate_call.allowed_target_claim_ids)}，',
        f'S={sorted(skeptic_call.allowed_target_claim_ids)}',
    )
    print(
        '6. 屏障后成对公开：',
        [(output.output_id, output.agent_role) for output in pair_outputs],
    )

    if judge_index is None:
        print('7. 开场结束后不立即裁决：先进入第 1 轮反驳。')
        continue

    judge_call = attribution_calls[judge_index]

    judge_output = next(
        output
        for output in attribution_repository.outputs
        if output.agent_role == 'JUDGE'
        and output.phase == judge_call.phase
        and output.round_number == judge_call.round_number
    )
    judge_calls.append((judge_call, judge_output, judge_action))
    next_step = '继续下一轮' if judge_action == 'NEXT_ROUND' else '生成最终报告'
    print('7. 随后进入 _run_judge_call()：')
    print('   Judge 可见历史 ID：', public_history_ids(judge_call))
    print('   Evidence 白名单数量：', len(judge_call.allowed_evidence_ids))
    print('   Claim 白名单：', sorted(judge_call.allowed_target_claim_ids))
    print('   是否允许继续下一轮：', judge_call.allow_next_round)
    print('   工装提交动作：', judge_action)
    print('   Judge 公开输出：', (judge_output.output_id, judge_output.agent_role))
    print('   流程去向：', next_step)


【过程】A/S pair 与后置 Judge 的完整流程

阶段：独立开场
1. 冻结公开历史 ID： []（独立开场）
2. 创建角色 invocation： ['ADVOCATE', 'SKEPTIC']
3. 两侧 history 完全相同： True
4. Evidence 白名单： A=2 条， S=2 条， 完全相同=True
5. Claim 白名单： A=[]， S=[]
6. 屏障后成对公开： [('output_1', 'ADVOCATE'), ('output_2', 'SKEPTIC')]
7. 开场结束后不立即裁决：先进入第 1 轮反驳。

阶段：阶段一：第 1 轮反驳与裁决
1. 冻结公开历史 ID： ['output_1', 'output_2']
2. 创建角色 invocation： ['ADVOCATE', 'SKEPTIC']
3. 两侧 history 完全相同： True
4. Evidence 白名单： A=2 条， S=2 条， 完全相同=True
5. Claim 白名单： A=['claim_001']， S=['claim_001']
6. 屏障后成对公开： [('output_3', 'ADVOCATE'), ('output_4', 'SKEPTIC')]
7. 随后进入 _run_judge_call()：
   Judge 可见历史 ID： ['output_1', 'output_2', 'output_3', 'output_4']
   Evidence 白名单数量： 2
   Claim 白名单： ['claim_001', 'claim_003']
   是否允许继续下一轮： True
   工装提交动作： NEXT_ROUND
   Judge 公开输出： ('output_5', 'JUDGE')
   流程去向： 继续下一轮

阶段：阶段二：第 2 轮反驳与裁决
1. 冻结公开历史 ID： ['output_1', 'output_2', 'output_3', 'output_4', 'output_5']
2. 创建角色 invocation： ['ADVOCATE', 'SKEPTIC']
3. 两侧 history 完全相同： True
4. Evidence 白名单： A=2 条

&emsp;&emsp;前一段只打印公开调用记录，不读取隐藏推理。下面再直接检查真实 Handler 方法中是否包含共享历史、并行任务、屏障、轮次门禁和后置裁决这些确定性控制机制。

In [14]:
# 直接读取真实 Handler 方法源码，只打印学员需要核对的机制存在性。
pair_source = inspect.getsource(type(handler)._run_position_pair)
pair_source_checks = {
    '先生成一份共享历史': 'history_before_pair = self._visible_history' in pair_source,
    '并行创建两个任务': 'asyncio.create_task' in pair_source,
    '等待双方 settled': 'asyncio.gather' in pair_source and 'return_exceptions=True' in pair_source,
    '通过成对命令发布': 'publish_position_pair' in pair_source,
}
print('\n【源码核对】_run_position_pair() 的四个关键机制')
for mechanism, present in pair_source_checks.items():
    print(f'- {mechanism}：{present}')


judge_source = inspect.getsource(type(handler)._run_judge_call)
judge_source_checks = {
    '读取本轮完整可见历史': 'visible_history = self._visible_history' in judge_source,
    '创建 JUDGE invocation': 'role="JUDGE"' in judge_source,
    '单独调用裁决 Agent': 'run_judgment' in judge_source,
    '只接受 NEXT_ROUND 或 REPORT': 'action not in {"NEXT_ROUND", "REPORT"}' in judge_source,
    '继续动作受轮次门禁限制': 'not invocation.allow_next_round' in judge_source,
    '发布一条裁决公开输出': 'publish_agent_output' in judge_source,
}
print('\n【源码核对】_run_judge_call() 的六个关键机制')
for mechanism, present in judge_source_checks.items():
    print(f'- {mechanism}：{present}')
print('【本段边界】确定性 Judge 动作来自工装；数据库原子回滚和真实模型行为需要其他证据证明。')


【源码核对】_run_position_pair() 的四个关键机制
- 先生成一份共享历史：True
- 并行创建两个任务：True
- 等待双方 settled：True
- 通过成对命令发布：True

【源码核对】_run_judge_call() 的六个关键机制
- 读取本轮完整可见历史：True
- 创建 JUDGE invocation：True
- 单独调用裁决 Agent：True
- 只接受 NEXT_ROUND 或 REPORT：True
- 继续动作受轮次门禁限制：True
- 发布一条裁决公开输出：True
【本段边界】确定性 Judge 动作来自工装；数据库原子回滚和真实模型行为需要其他证据证明。


&emsp;&emsp;先读八行总时间线中的英文：`OPENING`（独立开场）、`REBUTTAL`（反驳）、`JUDGMENT`（裁决）、`history`（当前可见的公开历史数量）、`next`（是否允许继续下一轮）。完整两轮应依次出现：

```text
1  ADVOCATE（支持方）  OPENING（开场）
2  SKEPTIC（质疑方）  OPENING（开场）
3  ADVOCATE（支持方）  REBUTTAL（第1轮反驳）
4  SKEPTIC（质疑方）  REBUTTAL（第1轮反驳）
5  JUDGE（裁决方）     JUDGMENT（第1轮裁决，可继续）
6  ADVOCATE（支持方）  REBUTTAL（第2轮反驳）
7  SKEPTIC（质疑方）  REBUTTAL（第2轮反驳）
8  JUDGE（裁决方）     JUDGMENT（第2轮裁决，不可继续）
```

&emsp;&emsp;随后代码把三次 Pair 分开打印。开场双方的 history 和 Claim 白名单都为空；第一轮双方共同看到 `output_1/2`，只能引用 `claim_001`；第二轮双方共同看到 `output_1～5`，只能引用 `claim_001/003`。每次 Pair 都只在屏障之后成对产生 2 条公开输出。

&emsp;&emsp;两次裁决也分别展开：第一次裁决看见 `output_1～4`，`allow_next_round=True`，提交 `NEXT_ROUND` 并生成 `output_5`；第二次裁决看见 `output_1～7`，`allow_next_round=False`，只能提交 `REPORT` 并生成 `output_8`。因此裁决方读到的是完整的一轮结果，而不是半轮结果。

&emsp;&emsp;最后两组 `inspect`（源码读取）检查只回答“真实 Handler 方法中是否存在这些控制机制”。四个 Pair 检查和六个 Judge 检查为 `True`（机制存在），不等于已经测量并发速度，也不等于真实 Provider（模型服务提供方）会稳定返回合格内容。

#### 7.3.3 第三段：集中检查拓扑、历史与结束门禁

In [15]:
"""对两轮调用记录做可见的拓扑与终态检查。"""
actual_topology = _topology(attribution_calls)                            # 投影为 role/phase/round 三元组。
expected_topology = [
    ('ADVOCATE', 'OPENING', None), ('SKEPTIC', 'OPENING', None),
    ('ADVOCATE', 'REBUTTAL', 1), ('SKEPTIC', 'REBUTTAL', 1),
    ('JUDGE', 'JUDGMENT', 1), ('ADVOCATE', 'REBUTTAL', 2),
    ('SKEPTIC', 'REBUTTAL', 2), ('JUDGE', 'JUDGMENT', 2),
]
opening_is_independent = (
    attribution_calls[0].context['debate_history'] == []
    and attribution_calls[1].context['debate_history'] == []
)                                                                         # 两次开场都看不到对方结果。

round_two_shares_history = (
    attribution_calls[5].context['debate_history']
    == attribution_calls[6].context['debate_history']
)                                                                         # 第二轮双方从同一公开快照出发。

all_pair_histories_match = all(
    advocate.context['debate_history'] == skeptic.context['debate_history']
    for advocate, skeptic, _ in position_pairs
)                                                                         # 三次 pair 的双方历史都一致。

all_pair_evidence_matches = all(
    advocate.allowed_evidence_ids == skeptic.allowed_evidence_ids
    for advocate, skeptic, _ in position_pairs
)                                                                         # 三次 pair 的 Evidence 白名单都一致。

all_pair_claims_match = all(
    advocate.allowed_target_claim_ids == skeptic.allowed_target_claim_ids
    for advocate, skeptic, _ in position_pairs
)                                                                         # 三次 pair 的 Claim 白名单都一致。

pair_output_counts = [len(outputs) for _, _, outputs in position_pairs]   # 每次 pair 应成对产生两条位置输出。

judge_history_counts = [
    len(call.context['debate_history']) for call, _, _ in judge_calls
]                                                                         # 两次 Judge 应分别看到 4 条和 7 条完整历史。
judge_actions = [action for _, _, action in judge_calls]                 # 第一轮继续，第二轮报告。
judge_next_permissions = [call.allow_next_round for call, _, _ in judge_calls]
judge_output_ids = [output.output_id for _, output, _ in judge_calls]

print('【输出】固定拓扑检查')
print('最终状态：', attribution_repository.run.status)
print('实际调用次数：', len(attribution_calls))
print('调用预算正确：', len(attribution_calls) == EXPECTED_CALLS)
print('角色顺序正确：', actual_topology == expected_topology)
print('两次开场相互独立：', opening_is_independent)
print('第二轮双方共享同一历史快照：', round_two_shares_history)

print()
print('三次 pair 的双方历史全部一致：', all_pair_histories_match)
print('三次 pair 的 Evidence 白名单全部一致：', all_pair_evidence_matches)
print('三次 pair 的 Claim 白名单全部一致：', all_pair_claims_match)
print('三次 pair 的公开输出数量：', pair_output_counts)
print('pair 的四个源码机制全部存在：', all(pair_source_checks.values()))

print()
print('Judge 调用次数：', len(judge_calls))
print('两次 Judge 可见历史数量：', judge_history_counts)
print('两次 Judge 提交动作：', judge_actions)
print('两次 Judge 的下一轮权限：', judge_next_permissions)
print('两次 Judge 公开输出 ID：', judge_output_ids)
print('Judge 的六个源码机制全部存在：', all(judge_source_checks.values()))
print('最后一次 Judge 还能继续下一轮：', attribution_calls[-1].allow_next_round)
print('报告摘要：', attribution_repository.completed[0].report['executive_summary'])
print('【本段边界】确定性工装证明应用层顺序，不证明真实模型的输出质量或延迟。')

【输出】固定拓扑检查
最终状态： COMPLETED
实际调用次数： 8
调用预算正确： True
角色顺序正确： True
两次开场相互独立： True
第二轮双方共享同一历史快照： True

三次 pair 的双方历史全部一致： True
三次 pair 的 Evidence 白名单全部一致： True
三次 pair 的 Claim 白名单全部一致： True
三次 pair 的公开输出数量： [2, 2, 2]
pair 的四个源码机制全部存在： True

Judge 调用次数： 2
两次 Judge 可见历史数量： [4, 7]
两次 Judge 提交动作： ['NEXT_ROUND', 'REPORT']
两次 Judge 的下一轮权限： [True, False]
两次 Judge 公开输出 ID： ['output_5', 'output_8']
Judge 的六个源码机制全部存在： True
最后一次 Judge 还能继续下一轮： False
报告摘要： 区域变化是主要直接贡献，机制仍待验证。
【本段边界】确定性工装证明应用层顺序，不证明真实模型的输出质量或延迟。


```text
【输出】固定拓扑检查
最终状态： COMPLETED
实际调用次数： 8
调用预算正确： True
角色顺序正确： True
两次开场相互独立： True
第二轮双方共享同一历史快照： True

三次 pair 的双方历史全部一致： True
三次 pair 的 Evidence 白名单全部一致： True
三次 pair 的 Claim 白名单全部一致： True
三次 pair 的公开输出数量： [2, 2, 2]
pair 的四个源码机制全部存在： True

Judge 调用次数： 2
两次 Judge 可见历史数量： [4, 7]
两次 Judge 提交动作： ['NEXT_ROUND', 'REPORT']
两次 Judge 的下一轮权限： [True, False]
两次 Judge 公开输出 ID： ['output_5', 'output_8']
Judge 的六个源码机制全部存在： True
最后一次 Judge 还能继续下一轮： False
报告摘要： 区域变化是主要直接贡献，机制仍待验证。
【本段边界】确定性工装证明应用层顺序，不证明真实模型的输出质量或延迟。
```

&emsp;&emsp;`COMPLETED`（已完成）表示这条确定性流程已经到达合法终态。所有布尔值为 `True`，共同证明本次保存的调用记录符合预期拓扑、同轮输入一致、裁决方后置并在末轮关闭继续权限；它们不证明报告里的业务解释已经构成因果事实。

&emsp;&emsp;本章 Notebook 的既有运行证据来自一次 fresh kernel（全新内核）完整执行：25 个代码单元连续完成且没有错误输出，未调用真实模型。相关定向 pytest（Python 测试框架）没有成功进入断言阶段，因此不能写成“定向测试已通过”；Fake Repository 也不能证明 PostgreSQL 数据库的成对写入与原子回滚。

### 7.4 一份冻结 Evidence（证据）怎样走到归因报告？

&emsp;&emsp;现在把代码输出还原成一条业务时间线。下面只使用 deterministic fixture（确定性测试夹具）中的公开调用记录，不连接真实模型，也不展示模型内部思维链。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 7-2　八次调用怎样逐步形成最终报告</font></p>

<div class="courseware-table-wrap">

| 阶段 | 角色看到什么 | 公开输出 | 流程为什么这样走 |
| --- | --- | --- | --- |
| 独立开场 | 支持方和质疑方的 history（公开历史）均为空；Claim（主张）白名单为空 | `output_1`、`output_2` | 两方先独立表达，不能读取对方尚未公开的开场结果 |
| 第 1 轮 A/S Pair | 双方共同看到 `output_1/2`，Evidence 白名单相同，只能引用 `claim_001` | `output_3`、`output_4` | 相同输入保证同轮公平；两方完成后才成对公开 |
| 第 1 轮 JUDGE（裁决方） | 读取 `output_1～4`，下一轮权限为 `True` | `output_5`，动作是 `NEXT_ROUND`（继续下一轮） | 裁决方认为分歧仍值得继续，`output_5` 也成为下一轮公开历史 |
| 第 2 轮 A/S Pair | 双方共同看到 `output_1～5`，只能引用 `claim_001/003` | `output_6`、`output_7` | 第二轮仍从同一公开快照出发，不能越过白名单引用其他主张 |
| 第 2 轮 JUDGE（裁决方） | 读取 `output_1～7`，下一轮权限为 `False` | `output_8`，动作是 `REPORT`（生成报告） | 已达到轮次上限，不能再创建第三轮，只能形成最终报告 |

</div>

&emsp;&emsp;最终报告摘要是“区域变化是主要直接贡献，机制仍待验证”。前半句说明当前 Evidence 支持什么，后半句保留因果边界。`output_8` 是受控流程的最终公开记录，但最终业务判断仍然交给人，而不是交给三个角色投票决定。

&emsp;&emsp;一句话收束：**冻结 Evidence → A/S 独立开场 → A/S 同轮配对反驳 → JUDGE 后置裁决 → 达到门禁后生成 REPORT**。

### 7.5 学习检查

1. 为什么 `max_rounds=2`（最多两轮）不等于一定执行 8 次角色调用？本章工装为什么恰好是 8 次？
2. 同一轮的 ADVOCATE（支持方）与 SKEPTIC（质疑方）为什么必须使用同一份 history（公开历史）快照？
3. JUDGE（裁决方）为什么必须等待双方 settled（均已结束）并成对发布后才能进入？
4. 第一轮裁决和第二轮裁决分别看见多少条公开历史？为什么下一轮权限不同？
5. 所有拓扑检查为 `True` 后，为什么仍不能声称真实模型归因和数据库事务已经通过？

<details><summary>参考要点</summary>

1. 开场固定 2 次，每个实际完成轮次固定增加 A/S/JUDGE 3 次；实际轮数可能小于上限。本章预设第一轮 `NEXT_ROUND`、第二轮 `REPORT`，所以完整执行 `2 + 3 × 2 = 8` 次。
2. 同一快照避免一方先返回后影响同轮另一方，保证双方在相同的 Evidence、历史和 Claim 白名单下独立形成位置。
3. 屏障保证裁决方看到完整的一轮公开记录，而不是基于半轮结果提前裁决。
4. 第一次裁决看见 4 条历史且 `allow_next_round=True`；第二次看见 7 条历史且已经到达上限，所以权限为 `False`。
5. 确定性工装只证明应用层顺序和门禁；它没有调用真实模型，也没有使用 PostgreSQL 验证成对写入、原子提交或回滚。

</details>

&emsp;&emsp;固定拓扑只能保证应用层按预定顺序调用角色，不能自动保证模型提交的 Evidence（证据）引用、action（动作）和 Claim（主张）指向合法。每次结构化提交还必须经过白名单、single slot（单提交槽位）和 server assembly（服务端装配），才能成为可信公开记录。

## <center>第八章：模型输出如何成为可信归因记录</center>

&emsp;&emsp;角色调用顺序明确后，还需要收束一个问题：模型提交的内容怎样经过服务器校验，才成为可追溯的公开记录。结构化 JSON（结构化数据）本身不可信，模型仍可能引用越权 Evidence（证据）、指向不可见 Claim（主张）、使用错误动作或重复提交。

&emsp;&emsp;本章不展开所有实现细节，只保留“白名单选择 → Publisher 校验 → Assembler 装配 → 公开状态”四步。学完后，你只需记住：模型负责提交业务内容，服务器负责生成可信坐标；结构可信不代表内容一定正确。

### 8.1 模型只提交业务内容，可信坐标由谁生成？

&emsp;&emsp;模型只提交业务陈述，并从服务器提供的白名单中选择 `evidence_id`（证据编号）与 `target_claim_id`（目标主张编号）。Publisher（提交校验器）检查调用 ID、允许动作、单提交槽位及 Evidence/Claim 白名单；Assembler（可信字段装配器）再由服务器补齐 `output_id`（公开输出编号）、角色、阶段、轮次和引用坐标。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164222659.png" alt="白名单校验与服务端可信装配" width="85%"></div>

&emsp;&emsp;图解：服务器先提供最小上下文与白名单；模型填写业务陈述并选择允许的 ID；Publisher 像安全门一样拒绝越权、错误动作和重复提交；Assembler 再装配服务器可信字段。

&emsp;&emsp;事实边界：用户背景继续标记为 `USER_PROVIDED_UNVERIFIED`（用户提供但未经数据验证）；完整 FactRef、指标定义和结果坐标留在服务器快照，不进入模型上下文。Publisher 只能证明提交结构与引用范围合法，不能判断模型的自然语言解释是否正确。源码锚点：[attribution.py](HermesAnalytics-第三版/backend/src/hermes_analytics/hermes_adapter/attribution.py)、[assembler.py](HermesAnalytics-第三版/backend/src/hermes_analytics/application/attribution/assembler.py)。

### 8.2 最小代码只需要观察哪三类结果？

&emsp;&emsp;Notebook 已把完整演示拆成“准备白名单、合法提交、四类拒绝、完全未提交”四段。下面保留一份同义的最小探针，学习时只观察三类结果：合法提交被接受；越权、错误动作和重复提交被拒绝；完全未提交返回稳定结构错误。它不调用模型或数据库。

In [16]:
from hermes_analytics.hermes_adapter.attribution import AttributionSubmissionPublisher
from hermes_analytics.hermes_adapter.errors import HermesAdapterError
from hermes_analytics.hermes_adapter.port import AttributionToolSubmission


def make_publisher() -> AttributionSubmissionPublisher:
    return AttributionSubmissionPublisher(
        invocation_id='call_1',
        expected_action='POSITION',
        allowed_evidence_ids=frozenset({'evidence_1'}),
        allowed_target_claim_ids=frozenset({'claim_1'}),
    )


def submission(*, action='POSITION', evidence='evidence_1', claim='claim_1'):
    return AttributionToolSubmission(
        invocation_id='call_1',
        action=action,
        payload={'evidence_ids': [evidence], 'target_claim_id': claim},
    )


accepted = make_publisher()
accepted.publish_submission(submission())
print('合法提交：', accepted.require_submission().action)

cases = (
    ('越权 Evidence', lambda p: p.publish_submission(submission(evidence='evidence_outside'))),
    ('越权 Claim', lambda p: p.publish_submission(submission(claim='claim_outside'))),
    ('错误动作', lambda p: p.publish_submission(submission(action='REPORT'))),
)
for label, attempt in cases:
    try:
        attempt(make_publisher())
    except PermissionError as error:
        print(label, '：已拒绝，', error)

duplicate = make_publisher()
duplicate.publish_submission(submission())
try:
    duplicate.publish_submission(submission())
except PermissionError as error:
    print('重复提交：已拒绝，', error)

try:
    make_publisher().require_submission()
except HermesAdapterError as error:
    print('完全未提交：已拒绝，', error.code.value)

合法提交： POSITION
越权 Evidence ：已拒绝， Evidence 不在本次冻结白名单中
越权 Claim ：已拒绝， Claim 不在当前可见公开历史中
错误动作 ：已拒绝， 当前归因角色不允许该提交动作
重复提交：已拒绝， 同一次 Agent 调用不得重复提交
完全未提交：已拒绝， INVALID_STRUCTURE


<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 8-1　最小探针只需读懂的三个结果</font></p>

<div class="courseware-table-wrap">

| 观察结果 | 说明什么 | 不说明什么 |
| --- | --- | --- |
| 合法 `POSITION` 被接受 | 调用 ID、动作、Evidence/Claim 选择均在当前白名单内 | 不表示自然语言解释正确 |
| 越权 Evidence、越权 Claim、错误动作、重复提交均被拒绝 | Publisher 按 fail-closed（失败即关闭）方式守住范围和单槽位 | 不表示数据库或完整归因已运行 |
| 完全未提交返回 `INVALID_STRUCTURE`（无效结构） | 普通文本 final 不能冒充正式业务提交 | 不表示真实模型一定会稳定调用工具 |

</div>

&emsp;&emsp;Assembler 不在这段探针中运行。它位于下一道服务器边界：只有 Publisher 接受的业务提交，才会由 Assembler 补齐可信公开字段。完整四段代码和逐项输出保留在课程 Notebook 中。

### 8.3 页面流程和状态只需要记住什么？

&emsp;&emsp;归因工作台可以压缩成六个动作：**新建草稿 → 填写问题与可选背景 → 选择至少两条 Evidence 和最大轮数 → 保存并启动 → 观察公开事件 → 审阅报告**。点击 start（启动）后，问题、背景、轮数和证据才被冻结；HTTP 202（请求已受理）不等于报告已经完成。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164224540.png" alt="归因状态机与 SSE 公开事件" width="85%"></div>

&emsp;&emsp;主状态从 `DRAFT`（草稿）推进到 `COMPLETED`（已完成），也可能进入 `FAILED`（失败）或 `CANCELLED`（已取消）。SSE（服务器推送事件）只发送阶段、公开输出、报告完成和终态，不发送隐藏思维链。

&emsp;&emsp;已有证据必须分层理解：19 个定向单元测试证明本地契约；1 个系统测试证明 FastAPI + 隔离 PostgreSQL + 确定性 fake agent（替身智能体）的主链；一次真实模型浏览器完成样本只证明当时那次配置成功，后续四次失败样本则说明外部模型可能不提交合格结构。以上证据都不能证明模型服务 SLA（服务等级承诺）、超时恢复或生产稳定性。

### 8.4 学习检查

1. 为什么模型只能选择 `evidence_id` 和 `target_claim_id`，不能自己生成角色、轮次与引用坐标？
2. Publisher（提交校验器）与 Assembler（可信字段装配器）分别负责什么？
3. 为什么普通文本 final 即使写得完整，也不能成为正式归因记录？
4. `19 passed` 与 `1 passed` 分别证明到哪一层？为什么仍不能声称真实模型多智能体 E2E（端到端）已经通过？

<details><summary>参考要点</summary>

1. 模型只负责业务陈述和白名单选择；服务器字段必须由确定性代码生成，避免模型伪造可信坐标。
2. Publisher 校验调用、动作、单槽位及 Evidence/Claim 白名单；Assembler 为合法提交补齐公开 ID、角色、阶段、轮次与引用。
3. 没有写入当前调用的结构化单槽位，就会以 `INVALID_STRUCTURE` 拒绝；普通文本不进入正式记录。
4. `19 passed` 证明本地确定性契约；`1 passed` 证明 FastAPI、隔离 PostgreSQL 与 fake agent 的系统主链。两者都没有证明真实模型输出质量、服务稳定性或生产验收。

</details>

&emsp;&emsp;提交成为可信结构后，还要回答“可信结构形成报告时，人应该怎样审阅并作出最终判断”。

## <center>第九章：人类如何审阅归因报告并准确表达项目</center>

&emsp;&emsp;系统生成报告只是技术工作流的终点，不是业务判断的终点。本章只解决两个问题：人类怎样审阅报告而不越过证据边界，以及怎样把三条技术链路准确地表达成一个项目闭环。

### 9.1 报告交给人类，而不是替人类决策

&emsp;&emsp;JUDGE（裁决方）的 `REPORT`（报告）不是“真相按钮”。Assembler（可信字段装配器）将模型提交的业务内容整理为主要驱动、抵消因素、拒绝解释、证据缺口与下一步建议。它帮助人类看清“当前证据支持到哪里”，不能把相关关系或贡献位置包装成已经证明的因果机制。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260826164225692.png" alt="从归因报告到人类业务判断" width="85%"></div>

&emsp;&emsp;图解：报告分解为主要驱动、抵消因素、拒绝解释、证据缺口和下一步建议；人类再结合业务约束做判断，缺口可回到补充分析。

&emsp;&emsp;即使报告给出高置信度，也不等于获得自动执行许可。报告不能独自证明因果关系，也不能独自给出精确贡献分解。可以继续对照 [Assembler 源码](HermesAnalytics-第三版/backend/src/hermes_analytics/application/attribution/assembler.py)、[归因 Operation 源码](HermesAnalytics-第三版/backend/src/hermes_analytics/application/attribution/operation.py) 和 [Operation 单元测试](HermesAnalytics-第三版/backend/tests/unit/test_attribution_operation.py) 检查这一边界。

### 9.2 人类审阅时要守住哪些边界？

- 冻结时点、指标口径和 Evidence 范围是否仍适用于当前业务问题？
- 每个主要驱动和抵消因素是否有冻结 Evidence 引用？
- 是否把贡献位置、相关关系与因果机制清楚分开？
- `REJECTED_EXPLANATION`（被拒绝的解释）是否明确写出为什么当前证据不足？
- `evidence_gaps`（证据缺口）与下一步建议是否足以支撑继续分析？
- 是否有外部约束、政策、成本、时效或业务目标没有进入这份证据？

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 9-1　归因分析的常见误区</font></p>

<div class="courseware-table-wrap">

| 误区 | 正确判断 |
| --- | --- |
| 三个角色意见一致就是因果结论 | 不是；JUDGE 不按票数裁决，证据仍可能只支持相关关系或贡献位置 |
| FactRef（事实引用）已冻结，所以模型看到了全部数据细节 | 不是；模型只看最小必要内容，完整 FactRef 留在冻结快照中负责追溯 |
| ADVOCATE（支持方）与 SKEPTIC（质疑方）并行，谁先返回谁就先发布 | 不是；双方都完成后才成对原子发布，JUDGE 随后运行 |
| JSON（结构化数据）格式合法就可信 | 不是；模型只能选择白名单编号，可信坐标仍由服务器校验和装配 |
| HTTP 202 表示报告已经完成 | 不是；它只表示请求已受理，还要等待终态、公开事件、报告和 operation（运行记录）闭环 |

</div>

### 9.3 怎样把三条链收束成一个项目闭环？

&emsp;&emsp;HermesAnalytics 的这一段闭环不是“让模型自由辩论”，而是：管理员的可信写链先让分析师获得新能力；分析完成后服务器冻结 Evidence；Hermes Agent 每次只承担一个受限角色；应用层固定支持方与质疑方并行、同步屏障和裁决方后置；模型只提交业务字段与白名单选择；服务器装配可信公开坐标；状态与报告对外可见；最终人类根据证据范围、缺口和业务约束作出判断。

&emsp;&emsp;这套设计的价值，是把模型擅长的解释竞争放进可审计边界，而不是把业务决定交给模型。对外可以用下面这段话概括项目：

> HermesAnalytics 先通过管理员可信写链发布新的分析能力，再把已完成分析冻结为 Evidence（证据）。归因应用固定支持方、质疑方并行和裁决方后置的流程；模型只选择白名单编号，服务器负责校验并装配可信坐标；报告只表达证据支持、被拒绝的解释和证据缺口，最终判断仍由人完成。

&emsp;&emsp;表达项目时还要把不同证据层分开，不能把低一层的通过结果冒充更高一层的验收。

<style>
.courseware-table-wrap{width:fit-content;max-width:100%;margin-left:auto;margin-right:auto;margin-bottom:1.25em;overflow-x:auto;}
.courseware-table-wrap table{margin-left:auto!important;margin-right:auto!important;}
.courseware-table-caption{text-align:center!important;width:100%;}
</style>

<p class="courseware-table-caption"><font face="黑体" size=4>表 9-2　项目证据层与表达边界</font></p>

<div class="courseware-table-wrap">

| 证据层 | 可以说明 | 不能据此说明 |
| --- | --- | --- |
| 确定性契约测试 | Publisher、Assembler、状态机等本地规则符合预期 | 数据库、真实模型或浏览器主链已经跑通 |
| FastAPI + PostgreSQL + fake agent（确定性替身智能体）系统测试 | 接口、隔离数据库与确定性替身可以完成系统主链 | 真实模型输出质量和 E2E（端到端）已经验证 |
| 一次真实模型浏览器样本 | 当时的源码、数据卷和配置曾形成一个可观察样本 | SLA（服务等级承诺）、超时恢复、生产稳定性已经验收 |
| 人工业务验收 | 人类已在指定环境按业务标准作出验收结论 | 不能由前面任一自动化证据自动替代 |

</div>

### 9.4 学习检查

1. 为什么 `REPORT` 不能被当作因果真相或自动决策？
2. 人类审阅主要驱动时，至少要核对哪三类边界？
3. 为什么 fake agent 系统测试通过，仍不能说明真实模型 E2E 和 SLA 已经验证？
4. 请用一句话串联“发布新能力 → 冻结 Evidence → 受控归因 → 人类判断”。

<details><summary>参考答案</summary>

1. 报告只整理当前冻结证据支持的解释、反证与缺口；相关关系和贡献位置不自动等于因果机制，高置信度也不是执行许可。
2. 至少核对冻结时点和指标口径是否适用、主要结论是否引用冻结 Evidence、相关与因果是否被混淆；还应检查拒绝理由、证据缺口和外部业务约束。
3. fake agent 的输出是确定性的，只能证明系统在预设响应下可以闭环；它没有覆盖真实模型的结构化提交稳定性、延迟、超时恢复和生产环境风险。
4. 先由管理员可信写链发布可验证的新分析能力，再把分析结果冻结为 Evidence；应用层让受限角色围绕同一证据运行，服务器校验和装配可信坐标，最后由人结合报告缺口与业务约束作出判断。

</details>